In [16]:
import subprocess
import librosa
import numpy as np
import json
from pytube import YouTube
import os
import time
import re
import urllib.parse as urlparse

# Sample JSON timestamps for the first recording
timestamps_json = '''
[{"t":0,"mix":0},{"t":2.929,"mix":1},{"t":4.756,"mix":2},{"t":6.573,"mix":3},{"t":8.281,"mix":4},{"t":9.939,"mix":5},{"t":11.639,"mix":6},{"t":13.259,"mix":7},{"t":14.866,"mix":8},{"t":16.63,"mix":9},{"t":18.354,"mix":10},{"t":20.097,"mix":11},{"t":21.898,"mix":12},{"t":23.611,"mix":13},{"t":25.245,"mix":14},{"t":26.811,"mix":15},{"t":28.725,"mix":16},{"t":30.656,"mix":17},{"t":32.361,"mix":18},{"t":34.086,"mix":19},{"t":35.798,"mix":20},{"t":37.684,"mix":21},{"t":39.434,"mix":22},{"t":41.233,"mix":23},{"t":43.176,"mix":24},{"t":44.828,"mix":25},{"t":46.555,"mix":26},{"t":48.472,"mix":27},{"t":50.301,"mix":28},{"t":51.989,"mix":29},{"t":53.736,"mix":30},{"t":55.688,"mix":31},{"t":57.531,"mix":32},{"t":59.37,"mix":33},{"t":61.042,"mix":34},{"t":62.78,"mix":35},{"t":64.472,"mix":36},{"t":66.137,"mix":37},{"t":67.761,"mix":38},{"t":69.611,"mix":39},{"t":71.43,"mix":40},{"t":72.976,"mix":41},{"t":74.685,"mix":42},{"t":76.401,"mix":43},{"t":77.973,"mix":44},{"t":79.532,"mix":45},{"t":81.078,"mix":46},{"t":82.721,"mix":47},{"t":84.438,"mix":48},{"t":86.027,"mix":49},{"t":87.755,"mix":50},{"t":89.656,"mix":51},{"t":91.38,"mix":52},{"t":93.045,"mix":53},{"t":94.735,"mix":54},{"t":96.353,"mix":55},{"t":97.959,"mix":56},{"t":99.453,"mix":57},{"t":101.119,"mix":58},{"t":102.738,"mix":59},{"t":104.287,"mix":60},{"t":105.84,"mix":61},{"t":107.435,"mix":62},{"t":109.056,"mix":63},{"t":110.694,"mix":64},{"t":112.215,"mix":65},{"t":113.829,"mix":66},{"t":115.528,"mix":67},{"t":117.136,"mix":68},{"t":118.636,"mix":69},{"t":120.23,"mix":70},{"t":121.821,"mix":71},{"t":123.451,"mix":72},{"t":125.183,"mix":73},{"t":126.896,"mix":74},{"t":128.36,"mix":75},{"t":130.109,"mix":76},{"t":131.703,"mix":77},{"t":133.343,"mix":78},{"t":134.977,"mix":79},{"t":136.778,"mix":80},{"t":138.437,"mix":81},{"t":140.239,"mix":82},{"t":141.849,"mix":83},{"t":143.508,"mix":84},{"t":145.086,"mix":85},{"t":146.744,"mix":86},{"t":148.254,"mix":87},{"t":149.915,"mix":88},{"t":151.593,"mix":89},{"t":153.248,"mix":90},{"t":154.873,"mix":91},{"t":156.448,"mix":92},{"t":158.193,"mix":93},{"t":159.765,"mix":94},{"t":161.582,"mix":95},{"t":163.306,"mix":96},{"t":164.79,"mix":97},{"t":166.423,"mix":98},{"t":167.982,"mix":99},{"t":169.527,"mix":100},{"t":171.032,"mix":101},{"t":172.816,"mix":102},{"t":174.655,"mix":103},{"t":176.149,"mix":104},{"t":177.75,"mix":105},{"t":179.522,"mix":106},{"t":181.175,"mix":107},{"t":182.915,"mix":108},{"t":184.644,"mix":109},{"t":186.38,"mix":110},{"t":188.043,"mix":111},{"t":189.83,"mix":112},{"t":191.548,"mix":113},{"t":193.126,"mix":114},{"t":194.902,"mix":115},{"t":196.502,"mix":116},{"t":198.194,"mix":117},{"t":199.803,"mix":118},{"t":201.301,"mix":119},{"t":203.038,"mix":120},{"t":204.775,"mix":121},{"t":206.417,"mix":122},{"t":208.046,"mix":123},{"t":209.74,"mix":124},{"t":211.336,"mix":125},{"t":212.891,"mix":126},{"t":214.628,"mix":127},{"t":216.307,"mix":128},{"t":217.907,"mix":129},{"t":219.524,"mix":130},{"t":221.101,"mix":131},{"t":222.835,"mix":132},{"t":224.415,"mix":133},{"t":226.115,"mix":134},{"t":227.782,"mix":135},{"t":229.369,"mix":136},{"t":230.983,"mix":137},{"t":232.67,"mix":138},{"t":234.342,"mix":139},{"t":235.978,"mix":140},{"t":237.608,"mix":141},{"t":239.286,"mix":142},{"t":240.951,"mix":143},{"t":242.539,"mix":144},{"t":244.097,"mix":145},{"t":245.762,"mix":146},{"t":247.454,"mix":147},{"t":249.096,"mix":148},{"t":250.795,"mix":149},{"t":252.524,"mix":150},{"t":254.119,"mix":151},{"t":255.718,"mix":152},{"t":257.22,"mix":153},{"t":258.839,"mix":154},{"t":260.359,"mix":155},{"t":262.005,"mix":156},{"t":263.633,"mix":157},{"t":265.339,"mix":158},{"t":267.135,"mix":159},{"t":268.872,"mix":160},{"t":270.451,"mix":161},{"t":272.069,"mix":162},{"t":273.766,"mix":163},{"t":275.386,"mix":164},{"t":277.005,"mix":165},{"t":278.82,"mix":166},{"t":280.521,"mix":167},{"t":282.144,"mix":168},{"t":283.751,"mix":169},{"t":285.522,"mix":170},{"t":287.285,"mix":171},{"t":288.886,"mix":172},{"t":290.482,"mix":173},{"t":292.387,"mix":174},{"t":294.211,"mix":175},{"t":295.794,"mix":176},{"t":297.382,"mix":177},{"t":299.226,"mix":178},{"t":301.093,"mix":179},{"t":302.749,"mix":180},{"t":304.326,"mix":181},{"t":305.962,"mix":182},{"t":307.618,"mix":183},{"t":309.321,"mix":184},{"t":310.766,"mix":185},{"t":312.286,"mix":186},{"t":313.753,"mix":187},{"t":315.299,"mix":188},{"t":316.821,"mix":189},{"t":318.279,"mix":190},{"t":320.022,"mix":191},{"t":321.424,"mix":192},{"t":322.978,"mix":193},{"t":324.626,"mix":194},{"t":326.792,"mix":195},{"t":328.593,"mix":196},{"t":330.288,"mix":197},{"t":332.191,"mix":198},{"t":333.765,"mix":199},{"t":335.348,"mix":200},{"t":336.92,"mix":201},{"t":338.498,"mix":202},{"t":340.098,"mix":203},{"t":341.562,"mix":204},{"t":343.054,"mix":205},{"t":344.726,"mix":206},{"t":346.344,"mix":207},{"t":347.896,"mix":208},{"t":349.525,"mix":209},{"t":351.157,"mix":210},{"t":352.666,"mix":211},{"t":354.257,"mix":212},{"t":356.507,"mix":213},{"t":358.356,"mix":214},{"t":359.952,"mix":215},{"t":361.697,"mix":216},{"t":363.338,"mix":217},{"t":364.852,"mix":218},{"t":366.317,"mix":219},{"t":367.826,"mix":220},{"t":369.332,"mix":221},{"t":370.885,"mix":222},{"t":372.485,"mix":223},{"t":374.017,"mix":224},{"t":375.576,"mix":225},{"t":377.073,"mix":226},{"t":378.723,"mix":227},{"t":380.176,"mix":228},{"t":381.802,"mix":229},{"t":383.333,"mix":230},{"t":384.868,"mix":231},{"t":386.429,"mix":232},{"t":387.949,"mix":233},{"t":389.607,"mix":234},{"t":390.996,"mix":235},{"t":392.605,"mix":236},{"t":394.164,"mix":237},{"t":395.711,"mix":238},{"t":397.249,"mix":239},{"t":398.789,"mix":240},{"t":400.293,"mix":241},{"t":401.894,"mix":242},{"t":403.395,"mix":243},{"t":404.972,"mix":244},{"t":406.483,"mix":245},{"t":408.069,"mix":246},{"t":409.567,"mix":247},{"t":411.092,"mix":248},{"t":412.648,"mix":249},{"t":414.185,"mix":250},{"t":415.745,"mix":251},{"t":417.312,"mix":252},{"t":418.803,"mix":253},{"t":420.267,"mix":254},{"t":421.895,"mix":255},{"t":423.463,"mix":256},{"t":425.024,"mix":257},{"t":426.484,"mix":258},{"t":428.206,"mix":259},{"t":429.794,"mix":260},{"t":431.428,"mix":261},{"t":432.986,"mix":262},{"t":434.659,"mix":263},{"t":436.196,"mix":264},{"t":437.781,"mix":265},{"t":439.373,"mix":266},{"t":440.852,"mix":267},{"t":442.382,"mix":268},{"t":443.89,"mix":269},{"t":445.378,"mix":270},{"t":447.049,"mix":271},{"t":448.56,"mix":272},{"t":450.164,"mix":273},{"t":451.871,"mix":274},{"t":453.604,"mix":275},{"t":455.154,"mix":276},{"t":456.723,"mix":277},{"t":458.277,"mix":278},{"t":459.827,"mix":279},{"t":461.53,"mix":280},{"t":463.238,"mix":281},{"t":464.718,"mix":282},{"t":466.404,"mix":283},{"t":468.008,"mix":284},{"t":469.499,"mix":285},{"t":471.011,"mix":286},{"t":472.671,"mix":287},{"t":474.404,"mix":288},{"t":476.012,"mix":289},{"t":477.558,"mix":290},{"t":479.25,"mix":291},{"t":480.87,"mix":292},{"t":482.45,"mix":293},{"t":484.091,"mix":294},{"t":485.619,"mix":295},{"t":487.121,"mix":296},{"t":488.594,"mix":297},{"t":490.181,"mix":298},{"t":491.678,"mix":299},{"t":493.321,"mix":300},{"t":495.021,"mix":301},{"t":496.706,"mix":302},{"t":498.347,"mix":303},{"t":500.034,"mix":304},{"t":501.704,"mix":305},{"t":503.299,"mix":306},{"t":505.05,"mix":307},{"t":506.808,"mix":308},{"t":508.506,"mix":309},{"t":510.138,"mix":310},{"t":511.754,"mix":311},{"t":513.291,"mix":312},{"t":514.759,"mix":313},{"t":516.35,"mix":314},{"t":518.041,"mix":315},{"t":519.75,"mix":316},{"t":521.42,"mix":317},{"t":522.999,"mix":318},{"t":524.586,"mix":319},{"t":526.27,"mix":320},{"t":527.919,"mix":321},{"t":529.526,"mix":322},{"t":531.23,"mix":323},{"t":532.906,"mix":324},{"t":534.531,"mix":325},{"t":536.258,"mix":326},{"t":537.926,"mix":327},{"t":539.496,"mix":328},{"t":541.157,"mix":329},{"t":542.801,"mix":330},{"t":544.51,"mix":331},{"t":546.066,"mix":332},{"t":547.677,"mix":333},{"t":549.156,"mix":334},{"t":550.756,"mix":335},{"t":552.318,"mix":336},{"t":554.031,"mix":337},{"t":555.533,"mix":338},{"t":557.266,"mix":339},{"t":558.921,"mix":340},{"t":560.583,"mix":341},{"t":562.012,"mix":342},{"t":563.705,"mix":343},{"t":565.528,"mix":344},{"t":567.244,"mix":345},{"t":568.979,"mix":346},{"t":570.552,"mix":347},{"t":572.236,"mix":348},{"t":573.949,"mix":349},{"t":575.51,"mix":350},{"t":577.291,"mix":351},{"t":578.996,"mix":352},{"t":580.544,"mix":353},{"t":582.22,"mix":354},{"t":583.884,"mix":355},{"t":585.567,"mix":356},{"t":587.301,"mix":357},{"t":588.886,"mix":358},{"t":590.673,"mix":359},{"t":592.162,"mix":360},{"t":593.703,"mix":361},{"t":595.39,"mix":362},{"t":597.222,"mix":363},{"t":598.714,"mix":364},{"t":600.414,"mix":365},{"t":601.831,"mix":366},{"t":603.499,"mix":367},{"t":604.84,"mix":368},{"t":606.587,"mix":369},{"t":608.134,"mix":370},{"t":610.022,"mix":371},{"t":611.524,"mix":372},{"t":613.259,"mix":373},{"t":614.95,"mix":374},{"t":616.481,"mix":375},{"t":618.266,"mix":376},{"t":620.12,"mix":377},{"t":621.739,"mix":378},{"t":623.558,"mix":379},{"t":625.186,"mix":380},{"t":626.84,"mix":381},{"t":628.517,"mix":382},{"t":630.21,"mix":383},{"t":631.822,"mix":384},{"t":633.389,"mix":385},{"t":635.022,"mix":386},{"t":636.682,"mix":387},{"t":638.288,"mix":388},{"t":639.976,"mix":389},{"t":641.486,"mix":390},{"t":643.088,"mix":391},{"t":644.558,"mix":392},{"t":646.233,"mix":393},{"t":647.817,"mix":394},{"t":649.278,"mix":395},{"t":650.997,"mix":396},{"t":652.774,"mix":397},{"t":654.401,"mix":398},{"t":655.894,"mix":399},{"t":657.39,"mix":400},{"t":659.108,"mix":401},{"t":660.698,"mix":402},{"t":662.467,"mix":403},{"t":664.061,"mix":404},{"t":665.716,"mix":405},{"t":667.356,"mix":406},{"t":669.033,"mix":407},{"t":670.835,"mix":408},{"t":672.437,"mix":409},{"t":674.085,"mix":410},{"t":675.607,"mix":411},{"t":677.296,"mix":412},{"t":678.753,"mix":413},{"t":680.335,"mix":414},{"t":681.95,"mix":415},{"t":683.653,"mix":416},{"t":685.263,"mix":417},{"t":686.888,"mix":418},{"t":688.628,"mix":419},{"t":690.238,"mix":420},{"t":691.791,"mix":421},{"t":693.393,"mix":422},{"t":695.008,"mix":423},{"t":696.635,"mix":424},{"t":698.194,"mix":425},{"t":699.862,"mix":426},{"t":701.615,"mix":427},{"t":703.249,"mix":428},{"t":704.872,"mix":429},{"t":706.591,"mix":430},{"t":708.211,"mix":431},{"t":709.781,"mix":432},{"t":711.405,"mix":433},{"t":713.027,"mix":434},{"t":714.612,"mix":435},{"t":716.248,"mix":436},{"t":717.859,"mix":437},{"t":719.407,"mix":438},{"t":721.098,"mix":439},{"t":722.742,"mix":440},{"t":724.364,"mix":441},{"t":725.945,"mix":442},{"t":727.512,"mix":443},{"t":729.045,"mix":444},{"t":730.753,"mix":445},{"t":732.225,"mix":446},{"t":733.843,"mix":447},{"t":735.476,"mix":448},{"t":737.011,"mix":449},{"t":738.501,"mix":450},{"t":739.927,"mix":451},{"t":741.522,"mix":452},{"t":743.292,"mix":453},{"t":744.766,"mix":454},{"t":746.349,"mix":455},{"t":748.055,"mix":456},{"t":749.788,"mix":457},{"t":751.288,"mix":458},{"t":752.929,"mix":459},{"t":754.405,"mix":460},{"t":755.98,"mix":461},{"t":757.395,"mix":462},{"t":759.068,"mix":463},{"t":760.761,"mix":464},{"t":762.415,"mix":465},{"t":763.985,"mix":466},{"t":765.553,"mix":467},{"t":767.135,"mix":468},{"t":768.698,"mix":469},{"t":770.483,"mix":470},{"t":772.08,"mix":471},{"t":773.743,"mix":472},{"t":775.229,"mix":473},{"t":776.86,"mix":474},{"t":778.546,"mix":475},{"t":780.163,"mix":476},{"t":781.857,"mix":477},{"t":783.46,"mix":478},{"t":785.047,"mix":479},{"t":786.678,"mix":480},{"t":788.287,"mix":481},{"t":789.983,"mix":482},{"t":791.676,"mix":483},{"t":793.291,"mix":484},{"t":794.962,"mix":485},{"t":796.549,"mix":486},{"t":798.236,"mix":487},{"t":799.863,"mix":488},{"t":801.492,"mix":489},{"t":803.17,"mix":490},{"t":804.851,"mix":491},{"t":806.52,"mix":492},{"t":808.215,"mix":493},{"t":809.708,"mix":494},{"t":811.264,"mix":495},{"t":812.871,"mix":496},{"t":814.533,"mix":497},{"t":816.092,"mix":498},{"t":817.892,"mix":499},{"t":819.48,"mix":500},{"t":821.147,"mix":501},{"t":822.705,"mix":502},{"t":824.209,"mix":503},{"t":825.917,"mix":504},{"t":827.528,"mix":505},{"t":829.713,"mix":506},{"t":831.648,"mix":507},{"t":833.41,"mix":508},{"t":835.244,"mix":509},{"t":837.919,"mix":510},{"t":840.265,"mix":511},{"t":842.954,"mix":512},{"t":844.591,"mix":513},{"t":846.572,"mix":514},{"t":848.468,"mix":515},{"t":850.252,"mix":516},{"t":852.197,"mix":517},{"t":854.071,"mix":518},{"t":855.875,"mix":519},{"t":857.777,"mix":520},{"t":859.549,"mix":521},{"t":861.408,"mix":522},{"t":863.171,"mix":523},{"t":864.831,"mix":524},{"t":866.595,"mix":525},{"t":868.218,"mix":526},{"t":869.966,"mix":527},{"t":871.734,"mix":528},{"t":873.339,"mix":529},{"t":875.015,"mix":530},{"t":876.861,"mix":531},{"t":878.551,"mix":532},{"t":880.252,"mix":533},{"t":882.061,"mix":534},{"t":883.811,"mix":535},{"t":885.57,"mix":536},{"t":887.237,"mix":537},{"t":888.98,"mix":538},{"t":890.969,"mix":539},{"t":892.559,"mix":540},{"t":894.137,"mix":541},{"t":895.782,"mix":542},{"t":897.505,"mix":543},{"t":898.997,"mix":544},{"t":900.763,"mix":545},{"t":902.674,"mix":546},{"t":905.29,"mix":547},{"t":908.689,"mix":548},{"t":909.279,"mix":549},{"t":909.864,"mix":550},{"t":910.397,"mix":551},{"t":910.896,"mix":552},{"t":911.572,"mix":553},{"t":912.038,"mix":554},{"t":912.487,"mix":555},{"t":912.915,"mix":556},{"t":913.401,"mix":557},{"t":913.905,"mix":558},{"t":914.407,"mix":559},{"t":914.891,"mix":560},{"t":915.351,"mix":561},{"t":915.83,"mix":562},{"t":916.343,"mix":563},{"t":916.788,"mix":564},{"t":917.257,"mix":565},{"t":917.793,"mix":566},{"t":918.306,"mix":567},{"t":918.72,"mix":568},{"t":919.199,"mix":569},{"t":919.699,"mix":570},{"t":920.155,"mix":571},{"t":920.619,"mix":572},{"t":921.11,"mix":573},{"t":921.587,"mix":574},{"t":922.059,"mix":575},{"t":922.534,"mix":576},{"t":923.007,"mix":577},{"t":923.509,"mix":578},{"t":923.981,"mix":579},{"t":924.478,"mix":580},{"t":924.919,"mix":581},{"t":925.401,"mix":582},{"t":925.924,"mix":583},{"t":926.371,"mix":584},{"t":926.811,"mix":585},{"t":927.278,"mix":586},{"t":927.719,"mix":587},{"t":928.187,"mix":588},{"t":928.658,"mix":589},{"t":929.164,"mix":590},{"t":929.654,"mix":591},{"t":930.116,"mix":592},{"t":930.597,"mix":593},{"t":931.077,"mix":594},{"t":931.56,"mix":595},{"t":932.027,"mix":596},{"t":932.436,"mix":597},{"t":932.9,"mix":598},{"t":933.375,"mix":599},{"t":933.867,"mix":600},{"t":934.327,"mix":601},{"t":934.774,"mix":602},{"t":935.27,"mix":603},{"t":935.784,"mix":604},{"t":936.321,"mix":605},{"t":936.818,"mix":606},{"t":937.32,"mix":607},{"t":937.802,"mix":608},{"t":938.297,"mix":609},{"t":938.77,"mix":610},{"t":939.265,"mix":611},{"t":939.758,"mix":612},{"t":940.224,"mix":613},{"t":940.733,"mix":614},{"t":941.205,"mix":615},{"t":941.672,"mix":616},{"t":942.173,"mix":617},{"t":942.636,"mix":618},{"t":943.093,"mix":619},{"t":943.563,"mix":620},{"t":944.039,"mix":621},{"t":944.549,"mix":622},{"t":944.993,"mix":623},{"t":945.547,"mix":624},{"t":946.052,"mix":625},{"t":946.603,"mix":626},{"t":947.127,"mix":627},{"t":947.667,"mix":628},{"t":948.149,"mix":629},{"t":948.729,"mix":630},{"t":949.261,"mix":631},{"t":949.807,"mix":632},{"t":950.357,"mix":633},{"t":950.885,"mix":634},{"t":951.348,"mix":635},{"t":951.858,"mix":636},{"t":952.375,"mix":637},{"t":952.872,"mix":638},{"t":953.414,"mix":639},{"t":953.986,"mix":640},{"t":954.475,"mix":641},{"t":955.029,"mix":642},{"t":955.553,"mix":643},{"t":956.085,"mix":644},{"t":956.593,"mix":645},{"t":957.077,"mix":646},{"t":957.611,"mix":647},{"t":958.17,"mix":648},{"t":958.668,"mix":649},{"t":959.165,"mix":650},{"t":959.712,"mix":651},{"t":960.24,"mix":652},{"t":960.705,"mix":653},{"t":961.178,"mix":654},{"t":961.694,"mix":655},{"t":962.261,"mix":656},{"t":962.795,"mix":657},{"t":963.284,"mix":658},{"t":963.802,"mix":659},{"t":964.326,"mix":660},{"t":964.802,"mix":661},{"t":965.283,"mix":662},{"t":965.78,"mix":663},{"t":966.413,"mix":664},{"t":966.949,"mix":665},{"t":967.47,"mix":666},{"t":967.968,"mix":667},{"t":968.347,"mix":668},{"t":968.828,"mix":669},{"t":969.401,"mix":670},{"t":969.928,"mix":671},{"t":970.412,"mix":672},{"t":970.895,"mix":673},{"t":971.426,"mix":674},{"t":971.949,"mix":675},{"t":972.456,"mix":676},{"t":972.912,"mix":677},{"t":973.444,"mix":678},{"t":973.924,"mix":679},{"t":974.416,"mix":680},{"t":974.906,"mix":681},{"t":975.474,"mix":682},{"t":975.976,"mix":683},{"t":976.505,"mix":684},{"t":977.014,"mix":685},{"t":977.52,"mix":686},{"t":978.029,"mix":687},{"t":978.514,"mix":688},{"t":979.014,"mix":689},{"t":979.534,"mix":690},{"t":980.125,"mix":691},{"t":980.627,"mix":692},{"t":981.151,"mix":693},{"t":981.715,"mix":694},{"t":982.221,"mix":695},{"t":982.732,"mix":696},{"t":983.191,"mix":697},{"t":983.708,"mix":698},{"t":984.22,"mix":699},{"t":984.739,"mix":700},{"t":985.24,"mix":701},{"t":985.834,"mix":702},{"t":986.336,"mix":703},{"t":986.79,"mix":704},{"t":987.221,"mix":705},{"t":987.911,"mix":706},{"t":988.415,"mix":707},{"t":988.948,"mix":708},{"t":989.451,"mix":709},{"t":989.98,"mix":710},{"t":990.479,"mix":711},{"t":990.969,"mix":712},{"t":991.486,"mix":713},{"t":992.008,"mix":714},{"t":992.494,"mix":715},{"t":992.992,"mix":716},{"t":993.517,"mix":717},{"t":994.011,"mix":718},{"t":994.527,"mix":719},{"t":995.039,"mix":720},{"t":995.552,"mix":721},{"t":996.063,"mix":722},{"t":997.665,"mix":723},{"t":998.594,"mix":724},{"t":999.143,"mix":725},{"t":999.631,"mix":726},{"t":1000.079,"mix":727},{"t":1000.615,"mix":728},{"t":1001.079,"mix":729},{"t":1001.554,"mix":730},{"t":1002.055,"mix":731},{"t":1002.5,"mix":732},{"t":1002.997,"mix":733},{"t":1003.503,"mix":734},{"t":1003.971,"mix":735},{"t":1004.439,"mix":736},{"t":1004.93,"mix":737},{"t":1005.437,"mix":738},{"t":1005.912,"mix":739},{"t":1006.396,"mix":740},{"t":1006.877,"mix":741},{"t":1007.425,"mix":742},{"t":1007.904,"mix":743},{"t":1008.347,"mix":744},{"t":1008.819,"mix":745},{"t":1009.353,"mix":746},{"t":1009.841,"mix":747},{"t":1010.317,"mix":748},{"t":1010.854,"mix":749},{"t":1011.311,"mix":750},{"t":1011.837,"mix":751},{"t":1012.334,"mix":752},{"t":1012.819,"mix":753},{"t":1013.333,"mix":754},{"t":1013.807,"mix":755},{"t":1014.272,"mix":756},{"t":1014.775,"mix":757},{"t":1015.278,"mix":758},{"t":1015.767,"mix":759},{"t":1016.235,"mix":760},{"t":1016.741,"mix":761},{"t":1017.237,"mix":762},{"t":1017.752,"mix":763},{"t":1018.237,"mix":764},{"t":1018.713,"mix":765},{"t":1019.215,"mix":766},{"t":1019.699,"mix":767},{"t":1020.172,"mix":768},{"t":1020.676,"mix":769},{"t":1021.154,"mix":770},{"t":1021.668,"mix":771},{"t":1022.152,"mix":772},{"t":1022.608,"mix":773},{"t":1023.114,"mix":774},{"t":1023.619,"mix":775},{"t":1024.106,"mix":776},{"t":1024.568,"mix":777},{"t":1025.052,"mix":778},{"t":1025.545,"mix":779},{"t":1026.011,"mix":780},{"t":1026.513,"mix":781},{"t":1026.977,"mix":782},{"t":1027.486,"mix":783},{"t":1028.004,"mix":784},{"t":1028.52,"mix":785},{"t":1028.997,"mix":786},{"t":1029.501,"mix":787},{"t":1029.975,"mix":788},{"t":1030.489,"mix":789},{"t":1031.02,"mix":790},{"t":1031.495,"mix":791},{"t":1032.002,"mix":792},{"t":1032.466,"mix":793},{"t":1032.965,"mix":794},{"t":1033.476,"mix":795},{"t":1034.001,"mix":796},{"t":1034.476,"mix":797},{"t":1034.946,"mix":798},{"t":1035.486,"mix":799},{"t":1036.018,"mix":800},{"t":1036.514,"mix":801},{"t":1036.996,"mix":802},{"t":1037.503,"mix":803},{"t":1038.021,"mix":804},{"t":1038.501,"mix":805},{"t":1038.941,"mix":806},{"t":1039.439,"mix":807},{"t":1039.931,"mix":808},{"t":1040.42,"mix":809},{"t":1040.863,"mix":810},{"t":1041.383,"mix":811},{"t":1041.884,"mix":812},{"t":1042.436,"mix":813},{"t":1043.008,"mix":814},{"t":1043.518,"mix":815},{"t":1044.105,"mix":816},{"t":1044.593,"mix":817},{"t":1045.137,"mix":818},{"t":1045.644,"mix":819},{"t":1046.181,"mix":820},{"t":1046.72,"mix":821},{"t":1047.228,"mix":822},{"t":1047.763,"mix":823},{"t":1048.29,"mix":824},{"t":1048.817,"mix":825},{"t":1049.309,"mix":826},{"t":1049.822,"mix":827},{"t":1050.337,"mix":828},{"t":1050.834,"mix":829},{"t":1051.297,"mix":830},{"t":1051.832,"mix":831},{"t":1052.301,"mix":832},{"t":1052.81,"mix":833},{"t":1053.265,"mix":834},{"t":1053.842,"mix":835},{"t":1054.31,"mix":836},{"t":1054.809,"mix":837},{"t":1055.306,"mix":838},{"t":1055.796,"mix":839},{"t":1056.321,"mix":840},{"t":1056.81,"mix":841},{"t":1057.324,"mix":842},{"t":1057.896,"mix":843},{"t":1058.487,"mix":844},{"t":1058.973,"mix":845},{"t":1059.472,"mix":846},{"t":1060.006,"mix":847},{"t":1060.546,"mix":848},{"t":1061.031,"mix":849},{"t":1061.569,"mix":850},{"t":1062.14,"mix":851},{"t":1062.689,"mix":852},{"t":1063.199,"mix":853},{"t":1063.777,"mix":854},{"t":1064.29,"mix":855},{"t":1064.813,"mix":856},{"t":1065.399,"mix":857},{"t":1065.906,"mix":858},{"t":1066.44,"mix":859},{"t":1066.989,"mix":860},{"t":1067.587,"mix":861},{"t":1068.121,"mix":862},{"t":1068.646,"mix":863},{"t":1069.218,"mix":864},{"t":1069.712,"mix":865},{"t":1070.286,"mix":866},{"t":1070.852,"mix":867},{"t":1071.377,"mix":868},{"t":1071.899,"mix":869},{"t":1072.439,"mix":870},{"t":1072.954,"mix":871},{"t":1073.513,"mix":872},{"t":1074.037,"mix":873},{"t":1074.55,"mix":874},{"t":1075.083,"mix":875},{"t":1075.632,"mix":876},{"t":1076.147,"mix":877},{"t":1076.702,"mix":878},{"t":1077.232,"mix":879},{"t":1077.739,"mix":880},{"t":1078.269,"mix":881},{"t":1078.803,"mix":882},{"t":1079.343,"mix":883},{"t":1079.861,"mix":884},{"t":1080.382,"mix":885},{"t":1080.912,"mix":886},{"t":1081.493,"mix":887},{"t":1081.992,"mix":888},{"t":1082.542,"mix":889},{"t":1083.054,"mix":890},{"t":1083.585,"mix":891},{"t":1084.117,"mix":892},{"t":1084.671,"mix":893},{"t":1085.201,"mix":894},{"t":1085.746,"mix":895},{"t":1086.247,"mix":896},{"t":1086.795,"mix":897},{"t":1087.299,"mix":898},{"t":1087.833,"mix":899},{"t":1088.317,"mix":900},{"t":1088.94,"mix":901},{"t":1089.517,"mix":902},{"t":1090.031,"mix":903},{"t":1090.504,"mix":904},{"t":1091.029,"mix":905},{"t":1091.559,"mix":906},{"t":1092.076,"mix":907},{"t":1092.6,"mix":908},{"t":1093.12,"mix":909},{"t":1093.629,"mix":910},{"t":1094.109,"mix":911},{"t":1094.596,"mix":912},{"t":1095.135,"mix":913},{"t":1095.615,"mix":914},{"t":1096.161,"mix":915},{"t":1096.656,"mix":916},{"t":1097.174,"mix":917},{"t":1097.671,"mix":918},{"t":1098.231,"mix":919},{"t":1098.749,"mix":920},{"t":1099.316,"mix":921},{"t":1099.826,"mix":922},{"t":1100.41,"mix":923},{"t":1100.909,"mix":924},{"t":1101.433,"mix":925},{"t":1101.992,"mix":926},{"t":1102.521,"mix":927},{"t":1103.071,"mix":928},{"t":1103.551,"mix":929},{"t":1104.063,"mix":930},{"t":1104.598,"mix":931},{"t":1105.146,"mix":932},{"t":1105.671,"mix":933},{"t":1106.318,"mix":946},{"t":1106.809,"mix":947},{"t":1107.265,"mix":948},{"t":1107.759,"mix":949},{"t":1108.274,"mix":950},{"t":1108.803,"mix":951},{"t":1109.342,"mix":952},{"t":1109.968,"mix":953},{"t":1112.285,"mix":954},{"t":1112.909,"mix":955},{"t":1113.356,"mix":956},{"t":1113.867,"mix":957},{"t":1114.363,"mix":958},{"t":1114.841,"mix":959},{"t":1115.375,"mix":960},{"t":1115.868,"mix":961},{"t":1116.318,"mix":962},{"t":1116.806,"mix":963},{"t":1117.276,"mix":964},{"t":1117.787,"mix":965},{"t":1118.234,"mix":966},{"t":1118.712,"mix":967},{"t":1119.18,"mix":968},{"t":1119.669,"mix":969},{"t":1120.169,"mix":970},{"t":1120.957,"mix":971},{"t":1121.79,"mix":972},{"t":1122.621,"mix":973},{"t":1123.446,"mix":974},{"t":1124.271,"mix":975},{"t":1125.019,"mix":976},{"t":1125.841,"mix":977},{"t":1126.681,"mix":978},{"t":1127.445,"mix":979},{"t":1128.171,"mix":980},{"t":1129.036,"mix":981},{"t":1129.753,"mix":974},{"t":1130.542,"mix":975},{"t":1131.318,"mix":976},{"t":1132.058,"mix":977},{"t":1132.846,"mix":978},{"t":1133.607,"mix":979},{"t":1134.331,"mix":982},{"t":1134.833,"mix":983},{"t":1135.236,"mix":984},{"t":1136.068,"mix":985},{"t":1136.962,"mix":986},{"t":1137.743,"mix":987},{"t":1138.604,"mix":988},{"t":1139.413,"mix":989},{"t":1140.283,"mix":990},{"t":1141.18,"mix":991},{"t":1142.061,"mix":992},{"t":1142.886,"mix":993},{"t":1143.776,"mix":994},{"t":1144.546,"mix":995},{"t":1145.393,"mix":996},{"t":1146.207,"mix":997},{"t":1147.095,"mix":998},{"t":1147.924,"mix":999},{"t":1148.804,"mix":1000},{"t":1149.704,"mix":1001},{"t":1150.535,"mix":1002},{"t":1151.324,"mix":1003},{"t":1152.182,"mix":1004},{"t":1153.005,"mix":1005},{"t":1153.841,"mix":1006},{"t":1154.626,"mix":1007},{"t":1155.487,"mix":1008},{"t":1156.317,"mix":1009},{"t":1157.133,"mix":1010},{"t":1157.945,"mix":1011},{"t":1158.746,"mix":1012},{"t":1159.593,"mix":1013},{"t":1160.404,"mix":1014},{"t":1161.209,"mix":1015},{"t":1162.099,"mix":1016},{"t":1162.885,"mix":1017},{"t":1163.672,"mix":1018},{"t":1164.446,"mix":1019},{"t":1165.33,"mix":1020},{"t":1166.108,"mix":1021},{"t":1166.972,"mix":1022},{"t":1167.775,"mix":1023},{"t":1168.656,"mix":1024},{"t":1169.43,"mix":1025},{"t":1170.304,"mix":1026},{"t":1171.081,"mix":1027},{"t":1171.909,"mix":1028},{"t":1172.758,"mix":1029},{"t":1173.631,"mix":1030},{"t":1174.394,"mix":1031},{"t":1175.167,"mix":1032},{"t":1176.079,"mix":1033},{"t":1176.872,"mix":1034},{"t":1177.72,"mix":1035},{"t":1178.507,"mix":1036},{"t":1179.407,"mix":1037},{"t":1180.206,"mix":1038},{"t":1180.917,"mix":1039},{"t":1181.683,"mix":1040},{"t":1182.51,"mix":1041},{"t":1183.361,"mix":1042},{"t":1184.133,"mix":1043},{"t":1184.896,"mix":1044},{"t":1185.729,"mix":1045},{"t":1186.535,"mix":1046},{"t":1187.282,"mix":1047},{"t":1188.129,"mix":1048},{"t":1188.65,"mix":1049},{"t":1189.727,"mix":1050},{"t":1190.509,"mix":1051},{"t":1191.272,"mix":1053},{"t":1192.172,"mix":1054},{"t":1193.011,"mix":1055},{"t":1193.806,"mix":1056},{"t":1194.619,"mix":1057},{"t":1195.44,"mix":1058},{"t":1196.261,"mix":1059},{"t":1197.024,"mix":1060},{"t":1197.869,"mix":1061},{"t":1198.719,"mix":1062},{"t":1199.498,"mix":1063},{"t":1200.32,"mix":1064},{"t":1201.131,"mix":1065},{"t":1202.026,"mix":1066},{"t":1202.845,"mix":1067},{"t":1203.672,"mix":1068},{"t":1204.52,"mix":1069},{"t":1205.451,"mix":1070},{"t":1206.26,"mix":1071},{"t":1207.079,"mix":1072},{"t":1207.999,"mix":1073},{"t":1208.873,"mix":1074},{"t":1209.713,"mix":1075},{"t":1210.577,"mix":1076},{"t":1211.404,"mix":1077},{"t":1212.331,"mix":1078},{"t":1213.209,"mix":1079},{"t":1214.089,"mix":1080},{"t":1214.949,"mix":1081},{"t":1215.867,"mix":1082},{"t":1216.749,"mix":1083},{"t":1217.617,"mix":1084},{"t":1218.526,"mix":1085},{"t":1219.462,"mix":1086},{"t":1220.346,"mix":1087},{"t":1221.466,"mix":1088},{"t":1222.762,"mix":1089},{"t":1223.938,"mix":1090},{"t":1225.188,"mix":1091},{"t":1226.764,"mix":1092},{"t":1231.53,"mix":1093},{"t":1232.4,"mix":1094},{"t":1232.985,"mix":1095},{"t":1233.546,"mix":1096},{"t":1234.115,"mix":1097},{"t":1234.659,"mix":1098},{"t":1235.228,"mix":1099},{"t":1235.76,"mix":1100},{"t":1236.217,"mix":1101},{"t":1236.733,"mix":1102},{"t":1237.288,"mix":1103},{"t":1237.812,"mix":1104},{"t":1238.303,"mix":1105},{"t":1238.8,"mix":1106},{"t":1239.251,"mix":1107},{"t":1239.736,"mix":1108},{"t":1240.195,"mix":1109},{"t":1240.656,"mix":1110},{"t":1241.17,"mix":1111},{"t":1241.622,"mix":1112},{"t":1242.091,"mix":1113},{"t":1242.627,"mix":1114},{"t":1243.095,"mix":1115},{"t":1243.555,"mix":1116},{"t":1244.026,"mix":1117},{"t":1244.556,"mix":1118},{"t":1245.004,"mix":1119},{"t":1245.489,"mix":1120},{"t":1245.95,"mix":1121},{"t":1246.456,"mix":1122},{"t":1246.948,"mix":1123},{"t":1247.405,"mix":1124},{"t":1247.818,"mix":1125},{"t":1248.31,"mix":1126},{"t":1248.79,"mix":1127},{"t":1249.245,"mix":1128},{"t":1249.705,"mix":1129},{"t":1250.185,"mix":1130},{"t":1250.637,"mix":1131},{"t":1251.121,"mix":1132},{"t":1251.567,"mix":1133},{"t":1252.04,"mix":1134},{"t":1252.512,"mix":1135},{"t":1252.945,"mix":1136},{"t":1253.391,"mix":1137},{"t":1253.887,"mix":1138},{"t":1254.398,"mix":1139},{"t":1254.895,"mix":1140},{"t":1255.358,"mix":1141},{"t":1255.834,"mix":1142},{"t":1256.305,"mix":1143},{"t":1256.751,"mix":1144},{"t":1257.225,"mix":1145},{"t":1257.671,"mix":1146},{"t":1258.163,"mix":1147},{"t":1258.644,"mix":1148},{"t":1259.124,"mix":1149},{"t":1259.641,"mix":1150},{"t":1260.133,"mix":1151},{"t":1260.634,"mix":1152},{"t":1261.157,"mix":1153},{"t":1261.647,"mix":1154},{"t":1262.14,"mix":1155},{"t":1262.623,"mix":1156},{"t":1263.084,"mix":1157},{"t":1263.606,"mix":1158},{"t":1264.079,"mix":1159},{"t":1264.534,"mix":1160},{"t":1265.023,"mix":1161},{"t":1265.518,"mix":1162},{"t":1266.017,"mix":1163},{"t":1266.521,"mix":1164},{"t":1267.017,"mix":1165},{"t":1267.497,"mix":1166},{"t":1267.989,"mix":1167},{"t":1268.476,"mix":1168},{"t":1268.971,"mix":1169},{"t":1269.507,"mix":1170},{"t":1270.054,"mix":1171},{"t":1270.517,"mix":1172},{"t":1271.046,"mix":1173},{"t":1271.647,"mix":1174},{"t":1272.17,"mix":1175},{"t":1272.711,"mix":1176},{"t":1273.186,"mix":1177},{"t":1273.748,"mix":1178},{"t":1274.304,"mix":1179},{"t":1274.844,"mix":1180},{"t":1275.367,"mix":1181},{"t":1275.83,"mix":1182},{"t":1276.342,"mix":1183},{"t":1276.828,"mix":1184},{"t":1277.325,"mix":1185},{"t":1277.827,"mix":1186},{"t":1278.356,"mix":1187},{"t":1278.912,"mix":1188},{"t":1279.404,"mix":1189},{"t":1279.98,"mix":1190},{"t":1280.488,"mix":1191},{"t":1280.998,"mix":1192},{"t":1281.497,"mix":1193},{"t":1282.02,"mix":1194},{"t":1282.536,"mix":1195},{"t":1283.011,"mix":1196},{"t":1283.53,"mix":1197},{"t":1284.045,"mix":1198},{"t":1284.601,"mix":1199},{"t":1285.07,"mix":1200},{"t":1285.583,"mix":1201},{"t":1286.112,"mix":1202},{"t":1286.62,"mix":1203},{"t":1287.156,"mix":1204},{"t":1287.664,"mix":1205},{"t":1288.155,"mix":1206},{"t":1288.667,"mix":1207},{"t":1289.193,"mix":1208},{"t":1289.673,"mix":1209},{"t":1290.213,"mix":1210},{"t":1290.741,"mix":1211},{"t":1291.271,"mix":1212},{"t":1291.78,"mix":1213},{"t":1292.346,"mix":1214},{"t":1292.774,"mix":1215},{"t":1293.3,"mix":1216},{"t":1293.815,"mix":1217},{"t":1294.342,"mix":1218},{"t":1294.797,"mix":1219},{"t":1295.3,"mix":1220},{"t":1295.751,"mix":1221},{"t":1296.285,"mix":1222},{"t":1296.785,"mix":1223},{"t":1297.264,"mix":1224},{"t":1297.764,"mix":1225},{"t":1298.271,"mix":1226},{"t":1298.748,"mix":1227},{"t":1299.256,"mix":1228},{"t":1299.767,"mix":1229},{"t":1300.286,"mix":1230},{"t":1300.848,"mix":1231},{"t":1301.357,"mix":1232},{"t":1301.897,"mix":1233},{"t":1302.359,"mix":1234},{"t":1302.873,"mix":1235},{"t":1303.408,"mix":1236},{"t":1303.974,"mix":1237},{"t":1304.485,"mix":1238},{"t":1305.025,"mix":1239},{"t":1305.603,"mix":1240},{"t":1306.158,"mix":1241},{"t":1306.612,"mix":1242},{"t":1306.993,"mix":1243},{"t":1307.491,"mix":1244},{"t":1308.048,"mix":1245},{"t":1308.534,"mix":1246},{"t":1309.068,"mix":1247},{"t":1309.653,"mix":1248},{"t":1310.181,"mix":1249},{"t":1310.673,"mix":1250},{"t":1311.083,"mix":1251},{"t":1311.703,"mix":1252},{"t":1312.261,"mix":1253},{"t":1312.793,"mix":1254},{"t":1313.296,"mix":1255},{"t":1313.804,"mix":1256},{"t":1314.31,"mix":1257},{"t":1314.822,"mix":1258},{"t":1315.341,"mix":1259},{"t":1315.832,"mix":1260},{"t":1316.349,"mix":1261},{"t":1316.873,"mix":1262},{"t":1317.402,"mix":1263},{"t":1317.94,"mix":1264},{"t":1318.441,"mix":1265},{"t":1318.926,"mix":1266},{"t":1319.444,"mix":1267},{"t":1319.946,"mix":1268},{"t":1321.512,"mix":1269},{"t":1322.506,"mix":1270},{"t":1323.012,"mix":1271},{"t":1323.511,"mix":1272},{"t":1323.994,"mix":1273},{"t":1324.498,"mix":1274},{"t":1324.947,"mix":1275},{"t":1325.42,"mix":1276},{"t":1325.904,"mix":1277},{"t":1326.364,"mix":1278},{"t":1326.849,"mix":1279},{"t":1327.323,"mix":1280},{"t":1327.807,"mix":1281},{"t":1328.308,"mix":1282},{"t":1328.769,"mix":1283},{"t":1329.253,"mix":1284},{"t":1329.735,"mix":1285},{"t":1330.225,"mix":1286},{"t":1330.672,"mix":1287},{"t":1331.176,"mix":1288},{"t":1331.684,"mix":1289},{"t":1332.167,"mix":1290},{"t":1332.661,"mix":1291},{"t":1333.186,"mix":1292},{"t":1333.657,"mix":1293},{"t":1334.185,"mix":1294},{"t":1334.706,"mix":1295},{"t":1335.169,"mix":1296},{"t":1335.678,"mix":1297},{"t":1336.17,"mix":1298},{"t":1336.653,"mix":1299},{"t":1337.149,"mix":1300},{"t":1337.611,"mix":1301},{"t":1338.124,"mix":1302},{"t":1338.626,"mix":1303},{"t":1339.074,"mix":1304},{"t":1339.567,"mix":1305},{"t":1340.062,"mix":1306},{"t":1340.606,"mix":1307},{"t":1341.039,"mix":1308},{"t":1341.575,"mix":1309},{"t":1342.065,"mix":1310},{"t":1342.549,"mix":1311},{"t":1343.037,"mix":1312},{"t":1343.496,"mix":1313},{"t":1343.966,"mix":1314},{"t":1344.459,"mix":1315},{"t":1344.927,"mix":1316},{"t":1345.397,"mix":1317},{"t":1345.881,"mix":1318},{"t":1346.413,"mix":1319},{"t":1346.884,"mix":1320},{"t":1347.374,"mix":1321},{"t":1347.893,"mix":1322},{"t":1348.357,"mix":1323},{"t":1348.854,"mix":1324},{"t":1349.358,"mix":1325},{"t":1349.877,"mix":1326},{"t":1350.361,"mix":1327},{"t":1350.861,"mix":1328},{"t":1351.339,"mix":1329},{"t":1351.816,"mix":1330},{"t":1352.328,"mix":1331},{"t":1352.818,"mix":1332},{"t":1353.303,"mix":1333},{"t":1353.804,"mix":1334},{"t":1354.3,"mix":1335},{"t":1354.82,"mix":1336},{"t":1355.314,"mix":1337},{"t":1355.803,"mix":1338},{"t":1356.287,"mix":1339},{"t":1356.775,"mix":1340},{"t":1357.29,"mix":1341},{"t":1357.755,"mix":1342},{"t":1358.293,"mix":1343},{"t":1358.778,"mix":1344},{"t":1359.251,"mix":1345},{"t":1359.769,"mix":1346},{"t":1360.271,"mix":1347},{"t":1360.75,"mix":1348},{"t":1361.271,"mix":1349},{"t":1361.786,"mix":1350},{"t":1362.273,"mix":1351},{"t":1362.756,"mix":1352},{"t":1363.256,"mix":1353},{"t":1363.748,"mix":1354},{"t":1364.243,"mix":1355},{"t":1364.72,"mix":1356},{"t":1365.249,"mix":1357},{"t":1365.733,"mix":1358},{"t":1366.249,"mix":1359},{"t":1366.775,"mix":1360},{"t":1367.323,"mix":1361},{"t":1367.895,"mix":1362},{"t":1368.426,"mix":1363},{"t":1368.979,"mix":1364},{"t":1369.473,"mix":1365},{"t":1370.001,"mix":1366},{"t":1370.509,"mix":1367},{"t":1371.031,"mix":1368},{"t":1371.568,"mix":1369},{"t":1372.071,"mix":1370},{"t":1372.584,"mix":1371},{"t":1373.058,"mix":1372},{"t":1373.589,"mix":1373},{"t":1374.082,"mix":1374},{"t":1374.583,"mix":1375},{"t":1375.101,"mix":1376},{"t":1375.611,"mix":1377},{"t":1376.144,"mix":1378},{"t":1376.637,"mix":1379},{"t":1377.148,"mix":1380},{"t":1377.648,"mix":1381},{"t":1378.144,"mix":1382},{"t":1378.681,"mix":1383},{"t":1379.157,"mix":1384},{"t":1379.662,"mix":1385},{"t":1380.153,"mix":1386},{"t":1380.644,"mix":1387},{"t":1381.144,"mix":1388},{"t":1381.693,"mix":1389},{"t":1382.253,"mix":1390},{"t":1382.79,"mix":1391},{"t":1383.293,"mix":1392},{"t":1383.832,"mix":1393},{"t":1384.35,"mix":1394},{"t":1384.894,"mix":1395},{"t":1385.419,"mix":1396},{"t":1386.025,"mix":1397},{"t":1386.568,"mix":1398},{"t":1387.062,"mix":1399},{"t":1387.617,"mix":1400},{"t":1388.124,"mix":1401},{"t":1388.704,"mix":1402},{"t":1389.229,"mix":1403},{"t":1389.761,"mix":1404},{"t":1390.296,"mix":1405},{"t":1390.855,"mix":1406},{"t":1391.396,"mix":1407},{"t":1391.992,"mix":1408},{"t":1392.529,"mix":1409},{"t":1393.056,"mix":1410},{"t":1393.566,"mix":1411},{"t":1394.12,"mix":1412},{"t":1394.657,"mix":1413},{"t":1395.18,"mix":1414},{"t":1395.692,"mix":1415},{"t":1396.232,"mix":1416},{"t":1396.789,"mix":1417},{"t":1397.311,"mix":1418},{"t":1397.829,"mix":1419},{"t":1398.334,"mix":1420},{"t":1398.87,"mix":1421},{"t":1399.499,"mix":1422},{"t":1400.014,"mix":1423},{"t":1400.546,"mix":1424},{"t":1401.102,"mix":1425},{"t":1401.633,"mix":1426},{"t":1402.187,"mix":1427},{"t":1402.722,"mix":1428},{"t":1403.246,"mix":1429},{"t":1403.767,"mix":1430},{"t":1404.316,"mix":1431},{"t":1404.841,"mix":1432},{"t":1405.365,"mix":1433},{"t":1405.869,"mix":1434},{"t":1406.416,"mix":1435},{"t":1406.884,"mix":1436},{"t":1407.439,"mix":1437},{"t":1407.969,"mix":1438},{"t":1408.502,"mix":1439},{"t":1409.003,"mix":1440},{"t":1409.56,"mix":1441},{"t":1410.089,"mix":1442},{"t":1410.624,"mix":1443},{"t":1411.163,"mix":1444},{"t":1411.682,"mix":1445},{"t":1412.163,"mix":1446},{"t":1412.805,"mix":1447},{"t":1413.33,"mix":1448},{"t":1413.859,"mix":1449},{"t":1414.441,"mix":1450},{"t":1414.955,"mix":1451},{"t":1415.415,"mix":1452},{"t":1415.964,"mix":1453},{"t":1416.485,"mix":1454},{"t":1416.976,"mix":1455},{"t":1417.417,"mix":1456},{"t":1417.95,"mix":1457},{"t":1418.453,"mix":1458},{"t":1418.983,"mix":1459},{"t":1419.467,"mix":1460},{"t":1420.036,"mix":1461},{"t":1420.511,"mix":1462},{"t":1421.01,"mix":1463},{"t":1421.508,"mix":1464},{"t":1422.045,"mix":1465},{"t":1422.573,"mix":1466},{"t":1423.105,"mix":1467},{"t":1423.684,"mix":1468},{"t":1424.24,"mix":1469},{"t":1424.747,"mix":1470},{"t":1425.3,"mix":1471},{"t":1425.82,"mix":1472},{"t":1426.308,"mix":1473},{"t":1426.891,"mix":1474},{"t":1427.412,"mix":1475},{"t":1427.965,"mix":1476},{"t":1428.526,"mix":1477},{"t":1429.094,"mix":1478},{"t":1429.59,"mix":1479},{"t":1430.031,"mix":1480},{"t":1430.596,"mix":1481},{"t":1431.153,"mix":1482},{"t":1431.661,"mix":1483},{"t":1432.204,"mix":1484},{"t":1432.675,"mix":1485},{"t":1433.199,"mix":1486},{"t":1433.721,"mix":1487},{"t":1436.033,"mix":1488},{"t":1436.598,"mix":1489},{"t":1437.078,"mix":1490},{"t":1437.593,"mix":1491},{"t":1438.099,"mix":1492},{"t":1438.597,"mix":1493},{"t":1439.128,"mix":1494},{"t":1439.594,"mix":1495},{"t":1440.081,"mix":1496},{"t":1440.538,"mix":1497},{"t":1441.013,"mix":1498},{"t":1441.473,"mix":1499},{"t":1441.966,"mix":1500},{"t":1442.404,"mix":1501},{"t":1442.883,"mix":1502},{"t":1443.315,"mix":1503},{"t":1443.738,"mix":1504},{"t":1444.534,"mix":1505},{"t":1445.376,"mix":1506},{"t":1446.211,"mix":1507},{"t":1447.029,"mix":1508},{"t":1447.752,"mix":1509},{"t":1448.544,"mix":1510},{"t":1449.319,"mix":1511},{"t":1450.105,"mix":1512},{"t":1450.831,"mix":1513},{"t":1451.901,"mix":1514},{"t":1452.823,"mix":1515},{"t":1453.57,"mix":1516},{"t":1457.543,"mix":1517},{"t":1466.699,"mix":1518},{"t":1475.553,"mix":1519},{"t":1482.007,"mix":1520},{"t":1487.44,"mix":1521},{"t":1493.875,"mix":1522},{"t":1501.205,"mix":1523},{"t":1508.192,"mix":1524},{"t":1515.605,"mix":1525},{"t":1523.239,"mix":1526},{"t":1530.565,"mix":1527},{"t":1537.969,"mix":1528},{"t":1544.3,"mix":1529},{"t":1551.106,"mix":1530},{"t":1557.992,"mix":1531},{"t":1565.139,"mix":1532},{"t":1571.83,"mix":1533},{"t":1578.664,"mix":1534},{"t":1585.585,"mix":1535},{"t":1591.726,"mix":1536},{"t":1598.079,"mix":1537},{"t":1604.549,"mix":1538},{"t":1611.408,"mix":1539},{"t":1618.641,"mix":1540},{"t":1625.542,"mix":1541},{"t":1630.101,"mix":1542},{"t":1634.331,"mix":1543},{"t":1638.373,"mix":1544},{"t":1642.689,"mix":1545},{"t":1646.716,"mix":1546},{"t":1650.709,"mix":1547},{"t":1654.875,"mix":1548},{"t":1659.071,"mix":1549},{"t":1662.77,"mix":1550},{"t":1666.94,"mix":1551},{"t":1670.935,"mix":1552},{"t":1675.115,"mix":1553},{"t":1679.331,"mix":1554},{"t":1683.174,"mix":1555},{"t":1687.509,"mix":1556},{"t":1692.076,"mix":1557},{"t":1696.555,"mix":1558},{"t":1705.596,"mix":1559},{"t":1711.683,"mix":1560},{"t":1717.112,"mix":1561},{"t":1722.189,"mix":1562},{"t":1727.769,"mix":1563},{"t":1733.596,"mix":1564},{"t":1739.081,"mix":1565},{"t":1744.622,"mix":1566},{"t":1750.084,"mix":1567},{"t":1755.558,"mix":1568},{"t":1761.563,"mix":1569},{"t":1766.886,"mix":1570},{"t":1772.294,"mix":1571},{"t":1778.243,"mix":1572},{"t":1783.629,"mix":1573},{"t":1789.43,"mix":1574},{"t":1795.037,"mix":1575},{"t":1800.81,"mix":1576},{"t":1806.713,"mix":1577},{"t":1812.889,"mix":1578},{"t":1819.475,"mix":1579},{"t":1826.319,"mix":1580},{"t":1832.974,"mix":1581},{"t":1837.055,"mix":1582},{"t":1840.782,"mix":1583},{"t":1844.752,"mix":1584},{"t":1848.434,"mix":1585},{"t":1852.135,"mix":1586},{"t":1855.924,"mix":1587},{"t":1859.928,"mix":1588},{"t":1864.081,"mix":1589},{"t":1868.173,"mix":1590},{"t":1871.947,"mix":1591},{"t":1875.756,"mix":1592},{"t":1879.844,"mix":1593},{"t":1883.726,"mix":1594},{"t":1887.565,"mix":1595},{"t":1891.696,"mix":1596},{"t":1895.835,"mix":1597},{"t":1900.463,"mix":1598},{"t":1906.335,"mix":1599},{"t":1912.296,"mix":1600},{"t":1918.393,"mix":1601},{"t":1924.601,"mix":1602},{"t":1930.789,"mix":1603},{"t":1936.743,"mix":1604},{"t":1942.609,"mix":1605},{"t":1948.667,"mix":1606},{"t":1955.033,"mix":1607},{"t":1961.208,"mix":1608},{"t":1967.129,"mix":1609},{"t":1973.487,"mix":1610},{"t":1979.713,"mix":1611},{"t":1985.891,"mix":1612},{"t":1992.409,"mix":1613},{"t":1998.835,"mix":1614},{"t":2005.409,"mix":1615},{"t":2011.825,"mix":1616},{"t":2018.062,"mix":1617},{"t":2024.356,"mix":1618},{"t":2030.741,"mix":1619},{"t":2037.211,"mix":1620},{"t":2043.725,"mix":1621},{"t":2050.075,"mix":1622},{"t":2056.685,"mix":1623},{"t":2063.285,"mix":1624},{"t":2069.812,"mix":1625},{"t":2076.322,"mix":1626},{"t":2082.777,"mix":1627},{"t":2089.824,"mix":1628},{"t":2096.518,"mix":1629},{"t":2103.319,"mix":1630},{"t":2109.979,"mix":1631},{"t":2117.005,"mix":1632},{"t":2123.228,"mix":1633},{"t":2130.143,"mix":1634},{"t":2137.123,"mix":1635},{"t":2144.071,"mix":1636},{"t":2151.062,"mix":1637},{"t":2157.855,"mix":1638},{"t":2164.735,"mix":1639},{"t":2171.776,"mix":1640},{"t":2178.711,"mix":1641},{"t":2185.821,"mix":1642},{"t":2192.475,"mix":1643},{"t":2199.055,"mix":1644},{"t":2205.373,"mix":1645},{"t":2212.283,"mix":1646},{"t":2219.035,"mix":1647},{"t":2226.313,"mix":1648},{"t":2233.566,"mix":1649},{"t":2240.8,"mix":1650},{"t":2248.091,"mix":1651},{"t":2255.868,"mix":1652},{"t":2264.175,"mix":1653},{"t":2270.912,"mix":1654},{"t":2278.271,"mix":1655},{"t":2285.695,"mix":1656},{"t":2292.793,"mix":1657},{"t":2299.549,"mix":1658},{"t":2306.215,"mix":1659},{"t":2312.732,"mix":1660},{"t":2319.222,"mix":1661},{"t":2325.988,"mix":1662},{"t":2332.663,"mix":1663},{"t":2340.088,"mix":1664},{"t":2348.217,"mix":1665},{"t":2356.669,"mix":1666},{"t":2365.695,"mix":1667},{"t":2375.26,"mix":1668},{"t":2384.654,"mix":1669},{"t":2394.009,"mix":1670},{"t":2403.762,"mix":1671},{"t":2413.429,"mix":1672},{"t":2423.586,"mix":1673},{"t":2435.466,"mix":1674},{"t":2436.836,"mix":1675},{"t":2437.89,"mix":1676},{"t":2438.79,"mix":1677},{"t":2439.675,"mix":1678},{"t":2440.541,"mix":1679},{"t":2441.458,"mix":1680},{"t":2442.359,"mix":1681},{"t":2443.241,"mix":1682},{"t":2444.363,"mix":1683},{"t":2445.504,"mix":1684},{"t":2447.054,"mix":1685},{"t":2448.281,"mix":1686},{"t":2449.488,"mix":1687},{"t":2451.064,"mix":1688},{"t":2452.691,"mix":1689},{"t":2454.807,"mix":1690},{"t":2456.449,"mix":1691},{"t":2457.608,"mix":1692},{"t":2458.603,"mix":1693},{"t":2459.442,"mix":1694},{"t":2460.414,"mix":1695},{"t":2461.288,"mix":1696},{"t":2462.224,"mix":1697},{"t":2463.252,"mix":1698},{"t":2464.472,"mix":1699},{"t":2465.925,"mix":1700},{"t":2467.595,"mix":1701},{"t":2468.937,"mix":1702},{"t":2470.903,"mix":1703},{"t":2473.826,"mix":1704},{"t":2475.071,"mix":1705},{"t":2476.742,"mix":1706},{"t":2478.468,"mix":1707},{"t":2479.85,"mix":1708},{"t":2481.219,"mix":1709},{"t":2482.999,"mix":1710},{"t":2484.421,"mix":1711},{"t":2486.029,"mix":1712},{"t":2487.522,"mix":1713},{"t":2489.124,"mix":1714},{"t":2490.505,"mix":1715},{"t":2492.186,"mix":1716},{"t":2493.802,"mix":1717},{"t":2495.908,"mix":1718},{"t":2498.839,"mix":1719},{"t":2500.737,"mix":1720},{"t":2503.004,"mix":1721},{"t":2504.865,"mix":1722},{"t":2505.502,"mix":1723},{"t":2505.95,"mix":1724},{"t":2506.434,"mix":1725},{"t":2506.886,"mix":1726},{"t":2507.374,"mix":1727},{"t":2507.823,"mix":1728},{"t":2508.296,"mix":1729},{"t":2509.271,"mix":1730},{"t":2510.808,"mix":1731},{"t":2512.244,"mix":1732},{"t":2514.084,"mix":1733},{"t":2516.359,"mix":1734},{"t":2518.765,"mix":1735},{"t":2522.004,"mix":1736},{"t":2525.322,"mix":1737},{"t":2530.365,"mix":1738},{"t":2535.554,"mix":1739},{"t":2537.161,"mix":1740},{"t":2539.237,"mix":1741},{"t":2540.813,"mix":1742},{"t":2542.923,"mix":1743},{"t":2544.645,"mix":1744},{"t":2546.379,"mix":1745},{"t":2548.438,"mix":1746},{"t":2550.334,"mix":1747},{"t":2552.238,"mix":1748},{"t":2554.76,"mix":1749},{"t":2557.598,"mix":1750},{"t":2560.696,"mix":1751},{"t":2562.285,"mix":1752},{"t":2563.542,"mix":1753},{"t":2564.918,"mix":1754},{"t":2566.334,"mix":1755},{"t":2568.14,"mix":1756},{"t":2569.789,"mix":1757},{"t":2571.515,"mix":1758},{"t":2573.503,"mix":1759},{"t":2575.787,"mix":1760},{"t":2577.602,"mix":1761},{"t":2579.317,"mix":1762},{"t":2581.544,"mix":1763},{"t":2584.498,"mix":1764},{"t":2587.054,"mix":1765},{"t":2590.276,"mix":1766},{"t":2592.403,"mix":1767},{"t":2594.032,"mix":1768},{"t":2595.616,"mix":1769},{"t":2597.186,"mix":1770},{"t":2598.951,"mix":1771},{"t":2600.631,"mix":1772},{"t":2602.162,"mix":1773},{"t":2603.784,"mix":1774},{"t":2605.505,"mix":1775},{"t":2607.09,"mix":1776},{"t":2608.713,"mix":1777},{"t":2610.468,"mix":1778},{"t":2612.311,"mix":1779},{"t":2613.927,"mix":1780},{"t":2615.538,"mix":1781},{"t":2617.362,"mix":1782},{"t":2618.951,"mix":1783},{"t":2620.619,"mix":1784},{"t":2622.205,"mix":1785},{"t":2624.027,"mix":1786},{"t":2625.782,"mix":1787},{"t":2627.402,"mix":1788},{"t":2629.024,"mix":1789},{"t":2630.686,"mix":1790},{"t":2632.587,"mix":1791},{"t":2634.279,"mix":1792},{"t":2635.868,"mix":1793},{"t":2637.387,"mix":1794},{"t":2639.084,"mix":1795},{"t":2640.622,"mix":1796},{"t":2642.244,"mix":1797},{"t":2643.993,"mix":1798},{"t":2645.508,"mix":1799},{"t":2647.107,"mix":1800},{"t":2648.694,"mix":1801},{"t":2650.43,"mix":1802},{"t":2652.118,"mix":1803},{"t":2653.674,"mix":1804},{"t":2655.257,"mix":1805},{"t":2656.956,"mix":1806},{"t":2658.559,"mix":1807},{"t":2660.249,"mix":1808},{"t":2661.807,"mix":1809},{"t":2663.588,"mix":1810},{"t":2665.349,"mix":1811},{"t":2666.91,"mix":1812},{"t":2668.533,"mix":1813},{"t":2670.221,"mix":1814},{"t":2671.826,"mix":1815},{"t":2673.356,"mix":1816},{"t":2674.986,"mix":1817},{"t":2676.636,"mix":1818},{"t":2678.382,"mix":1819},{"t":2679.958,"mix":1820},{"t":2681.49,"mix":1821},{"t":2683.161,"mix":1822},{"t":2684.81,"mix":1823},{"t":2686.504,"mix":1824},{"t":2688.095,"mix":1825},{"t":2689.817,"mix":1826},{"t":2691.5,"mix":1827},{"t":2693.085,"mix":1828},{"t":2694.64,"mix":1829},{"t":2696.368,"mix":1830},{"t":2698.018,"mix":1831},{"t":2699.688,"mix":1832},{"t":2701.224,"mix":1833},{"t":2702.946,"mix":1834},{"t":2704.488,"mix":1835},{"t":2705.944,"mix":1836},{"t":2707.482,"mix":1837},{"t":2709.128,"mix":1838},{"t":2710.765,"mix":1839},{"t":2712.313,"mix":1840},{"t":2713.934,"mix":1841},{"t":2715.561,"mix":1842},{"t":2717.105,"mix":1843},{"t":2718.636,"mix":1844},{"t":2720.16,"mix":1845},{"t":2721.757,"mix":1846},{"t":2723.312,"mix":1847},{"t":2724.898,"mix":1848},{"t":2726.464,"mix":1849},{"t":2728.092,"mix":1850},{"t":2729.611,"mix":1851},{"t":2731.188,"mix":1852},{"t":2732.748,"mix":1853},{"t":2734.321,"mix":1854},{"t":2735.909,"mix":1855},{"t":2737.512,"mix":1856},{"t":2739.008,"mix":1857},{"t":2740.623,"mix":1858},{"t":2742.113,"mix":1859},{"t":2743.698,"mix":1860},{"t":2745.209,"mix":1861},{"t":2746.764,"mix":1862},{"t":2748.292,"mix":1863},{"t":2749.8,"mix":1864},{"t":2751.389,"mix":1865},{"t":2752.838,"mix":1866},{"t":2754.38,"mix":1867},{"t":2755.86,"mix":1868},{"t":2757.491,"mix":1869},{"t":2759.001,"mix":1870},{"t":2760.569,"mix":1871},{"t":2762.114,"mix":1872},{"t":2763.692,"mix":1873},{"t":2765.139,"mix":1874},{"t":2766.656,"mix":1875},{"t":2768.18,"mix":1876},{"t":2770.25,"mix":1877},{"t":2773.599,"mix":1878},{"t":2777.894,"mix":1879},{"t":2782.326,"mix":1880},{"t":2783.977,"mix":1881},{"t":2785.548,"mix":1882},{"t":2786.901,"mix":1883},{"t":2787.739,"mix":1884},{"t":2788.549,"mix":1885},{"t":2789.412,"mix":1886},{"t":2790.287,"mix":1887},{"t":2791.12,"mix":1888},{"t":2791.899,"mix":1889},{"t":2792.558,"mix":1890},{"t":2793.757,"mix":1891},{"t":2794.918,"mix":1892},{"t":2795.994,"mix":1893},{"t":2797.916,"mix":1894},{"t":2799.922,"mix":1895},{"t":2801.675,"mix":1896},{"t":2802.999,"mix":1897},{"t":2804.845,"mix":1898},{"t":2806.763,"mix":1899},{"t":2808.012,"mix":1900},{"t":2809.388,"mix":1901},{"t":2810.664,"mix":1902},{"t":2812.037,"mix":1903},{"t":2813.537,"mix":1904},{"t":2814.947,"mix":1905},{"t":2817.902,"mix":1906},{"t":2819.087,"mix":1907},{"t":2820.272,"mix":1908},{"t":2822.988,"mix":1909},{"t":2825.737,"mix":1910},{"t":2830.01,"mix":1911},{"t":2831.414,"mix":1912},{"t":2833.218,"mix":1913},{"t":2834.779,"mix":1914},{"t":2836.522,"mix":1915},{"t":2838.138,"mix":1916},{"t":2839.781,"mix":1917},{"t":2841.405,"mix":1918},{"t":2842.956,"mix":1919},{"t":2844.534,"mix":1920},{"t":2846.082,"mix":1921},{"t":2847.688,"mix":1922},{"t":2849.23,"mix":1923},{"t":2850.755,"mix":1924},{"t":2852.331,"mix":1925},{"t":2853.848,"mix":1926},{"t":2855.506,"mix":1927},{"t":2857.011,"mix":1928},{"t":2858.567,"mix":1929},{"t":2860.129,"mix":1930},{"t":2861.86,"mix":1931},{"t":2863.427,"mix":1932},{"t":2865.015,"mix":1933},{"t":2866.637,"mix":1934},{"t":2868.263,"mix":1935},{"t":2869.856,"mix":1936},{"t":2871.413,"mix":1937},{"t":2872.977,"mix":1938},{"t":2874.655,"mix":1939},{"t":2876.233,"mix":1940},{"t":2877.897,"mix":1941},{"t":2879.566,"mix":1942},{"t":2881.176,"mix":1943},{"t":2882.849,"mix":1944},{"t":2884.489,"mix":1945},{"t":2886.037,"mix":1946},{"t":2887.652,"mix":1947},{"t":2889.269,"mix":1948},{"t":2890.902,"mix":1949},{"t":2892.47,"mix":1950},{"t":2894.129,"mix":1951},{"t":2895.781,"mix":1952},{"t":2897.415,"mix":1953},{"t":2899.064,"mix":1954},{"t":2900.808,"mix":1955},{"t":2902.275,"mix":1956},{"t":2904.036,"mix":1957},{"t":2905.766,"mix":1958},{"t":2907.479,"mix":1959},{"t":2909.257,"mix":1960},{"t":2910.842,"mix":1961},{"t":2912.547,"mix":1962},{"t":2914.153,"mix":1963},{"t":2915.845,"mix":1964},{"t":2917.508,"mix":1965},{"t":2919.15,"mix":1966},{"t":2920.805,"mix":1967},{"t":2922.466,"mix":1968},{"t":2924.187,"mix":1969},{"t":2925.748,"mix":1970},{"t":2927.404,"mix":1971},{"t":2929.032,"mix":1972},{"t":2930.581,"mix":1973},{"t":2932.18,"mix":1974},{"t":2933.871,"mix":1975},{"t":2935.473,"mix":1976},{"t":2937.044,"mix":1977},{"t":2938.681,"mix":1978},{"t":2940.395,"mix":1979},{"t":2942.033,"mix":1980},{"t":2943.677,"mix":1981},{"t":2945.276,"mix":1982},{"t":2947.015,"mix":1983},{"t":2948.629,"mix":1984},{"t":2950.231,"mix":1985},{"t":2951.965,"mix":1986},{"t":2953.706,"mix":1987},{"t":2955.344,"mix":1988},{"t":2956.919,"mix":1989},{"t":2958.535,"mix":1990},{"t":2960.133,"mix":1991},{"t":2961.756,"mix":1992},{"t":2963.286,"mix":1993},{"t":2964.924,"mix":1994},{"t":2966.655,"mix":1995},{"t":2968.309,"mix":1996},{"t":2970.07,"mix":1997},{"t":2971.841,"mix":1998},{"t":2973.46,"mix":1999},{"t":2975.228,"mix":2000},{"t":2976.963,"mix":2001},{"t":2978.701,"mix":2002},{"t":2980.415,"mix":2003},{"t":2983.273,"mix":2004},{"t":2992.035,"mix":2005},{"t":2993.888,"mix":2006},{"t":2994.904,"mix":2007},{"t":2995.836,"mix":2008},{"t":2996.761,"mix":2009},{"t":2997.834,"mix":2010},{"t":2998.912,"mix":2011},{"t":2999.98,"mix":2012},{"t":3000.991,"mix":2013},{"t":3001.994,"mix":2014},{"t":3003.003,"mix":2015},{"t":3004.011,"mix":2016},{"t":3004.961,"mix":2017},{"t":3005.951,"mix":2018},{"t":3006.9,"mix":2019},{"t":3007.901,"mix":2020},{"t":3008.848,"mix":2021},{"t":3009.822,"mix":2022},{"t":3010.785,"mix":2023},{"t":3011.754,"mix":2024},{"t":3012.69,"mix":2025},{"t":3013.712,"mix":2026},{"t":3014.68,"mix":2027},{"t":3015.621,"mix":2028},{"t":3016.534,"mix":2029},{"t":3017.582,"mix":2030},{"t":3018.501,"mix":2031},{"t":3019.371,"mix":2032},{"t":3020.314,"mix":2033},{"t":3021.316,"mix":2034},{"t":3022.271,"mix":2035},{"t":3023.221,"mix":2036},{"t":3024.111,"mix":2037},{"t":3025.038,"mix":2038},{"t":3025.977,"mix":2039},{"t":3026.919,"mix":2040},{"t":3027.832,"mix":2041},{"t":3028.802,"mix":2042},{"t":3029.703,"mix":2043},{"t":3030.697,"mix":2044},{"t":3031.646,"mix":2045},{"t":3032.592,"mix":2046},{"t":3033.54,"mix":2047},{"t":3034.413,"mix":2048},{"t":3035.402,"mix":2049},{"t":3036.413,"mix":2050},{"t":3037.419,"mix":2051},{"t":3038.401,"mix":2052},{"t":3039.38,"mix":2053},{"t":3040.415,"mix":2054},{"t":3041.408,"mix":2055},{"t":3042.408,"mix":2056},{"t":3043.408,"mix":2057},{"t":3044.404,"mix":2058},{"t":3045.435,"mix":2059},{"t":3046.447,"mix":2060},{"t":3047.422,"mix":2061},{"t":3048.465,"mix":2062},{"t":3049.447,"mix":2063},{"t":3050.419,"mix":2064},{"t":3051.437,"mix":2065},{"t":3052.452,"mix":2066},{"t":3053.491,"mix":2067},{"t":3054.508,"mix":2068},{"t":3055.523,"mix":2069},{"t":3056.476,"mix":2070},{"t":3057.441,"mix":2071},{"t":3058.442,"mix":2072},{"t":3059.506,"mix":2073},{"t":3060.528,"mix":2074},{"t":3061.512,"mix":2075},{"t":3062.524,"mix":2076},{"t":3063.485,"mix":2077},{"t":3064.526,"mix":2078},{"t":3065.474,"mix":2079},{"t":3066.48,"mix":2080},{"t":3067.481,"mix":2081},{"t":3068.507,"mix":2082},{"t":3069.557,"mix":2083},{"t":3070.536,"mix":2084},{"t":3071.445,"mix":2085},{"t":3072.534,"mix":2086},{"t":3073.539,"mix":2087},{"t":3074.498,"mix":2088},{"t":3075.467,"mix":2089},{"t":3076.464,"mix":2090},{"t":3077.479,"mix":2091},{"t":3078.458,"mix":2092},{"t":3079.408,"mix":2093},{"t":3080.422,"mix":2094},{"t":3081.362,"mix":2095},{"t":3082.281,"mix":2096},{"t":3083.226,"mix":2097},{"t":3084.218,"mix":2098},{"t":3085.153,"mix":2099},{"t":3086.113,"mix":2100},{"t":3087.1,"mix":2101},{"t":3088.124,"mix":2102},{"t":3089,"mix":2103},{"t":3089.889,"mix":2104},{"t":3090.774,"mix":2105},{"t":3091.633,"mix":2106},{"t":3092.581,"mix":2107},{"t":3093.398,"mix":2108},{"t":3094.223,"mix":2109},{"t":3095.028,"mix":2110},{"t":3095.871,"mix":2111},{"t":3096.688,"mix":2112},{"t":3097.492,"mix":2113},{"t":3098.302,"mix":2114},{"t":3099.137,"mix":2115},{"t":3099.951,"mix":2116},{"t":3100.793,"mix":2117},{"t":3101.611,"mix":2118},{"t":3102.444,"mix":2119},{"t":3103.275,"mix":2120},{"t":3104.089,"mix":2121},{"t":3104.94,"mix":2122},{"t":3105.734,"mix":2123},{"t":3106.573,"mix":2124},{"t":3107.454,"mix":2125},{"t":3108.23,"mix":2126},{"t":3109.037,"mix":2127},{"t":3109.863,"mix":2128},{"t":3110.68,"mix":2129},{"t":3111.513,"mix":2130},{"t":3112.366,"mix":2131},{"t":3113.231,"mix":2132},{"t":3114.046,"mix":2133},{"t":3114.917,"mix":2134},{"t":3115.776,"mix":2135},{"t":3116.588,"mix":2136},{"t":3117.449,"mix":2137},{"t":3118.299,"mix":2138},{"t":3119.165,"mix":2139},{"t":3120.045,"mix":2140},{"t":3120.894,"mix":2141},{"t":3121.687,"mix":2142},{"t":3122.454,"mix":2143},{"t":3123.285,"mix":2144},{"t":3124.134,"mix":2145},{"t":3124.948,"mix":2146},{"t":3125.786,"mix":2147},{"t":3126.599,"mix":2148},{"t":3127.435,"mix":2149},{"t":3128.259,"mix":2150},{"t":3129.123,"mix":2151},{"t":3129.922,"mix":2152},{"t":3130.722,"mix":2153},{"t":3131.557,"mix":2154},{"t":3132.361,"mix":2155},{"t":3133.186,"mix":2156},{"t":3133.974,"mix":2157},{"t":3134.734,"mix":2158},{"t":3135.573,"mix":2159},{"t":3136.426,"mix":2160},{"t":3137.247,"mix":2161},{"t":3138.05,"mix":2162},{"t":3138.83,"mix":2163},{"t":3139.648,"mix":2164},{"t":3140.474,"mix":2165},{"t":3141.318,"mix":2166},{"t":3142.142,"mix":2167},{"t":3142.981,"mix":2168},{"t":3143.811,"mix":2169},{"t":3144.602,"mix":2170},{"t":3145.451,"mix":2171},{"t":3146.266,"mix":2172},{"t":3147.102,"mix":2173},{"t":3147.939,"mix":2174},{"t":3148.787,"mix":2175},{"t":3149.603,"mix":2176},{"t":3150.421,"mix":2177},{"t":3151.26,"mix":2178},{"t":3152.135,"mix":2179},{"t":3152.962,"mix":2180},{"t":3153.77,"mix":2181},{"t":3154.627,"mix":2182},{"t":3155.46,"mix":2183},{"t":3156.362,"mix":2184},{"t":3157.168,"mix":2185},{"t":3157.952,"mix":2186},{"t":3158.789,"mix":2187},{"t":3159.636,"mix":2188},{"t":3160.455,"mix":2189},{"t":3161.319,"mix":2190},{"t":3162.176,"mix":2191},{"t":3163.138,"mix":2192},{"t":3164.13,"mix":2193},{"t":3165.045,"mix":2194},{"t":3165.967,"mix":2195},{"t":3166.873,"mix":2196},{"t":3167.833,"mix":2197},{"t":3168.786,"mix":2198},{"t":3169.686,"mix":2199},{"t":3170.786,"mix":2200},{"t":3171.819,"mix":2201},{"t":3172.79,"mix":2202},{"t":3173.86,"mix":2203},{"t":3174.867,"mix":2204},{"t":3175.885,"mix":2205},{"t":3176.984,"mix":2206},{"t":3178.133,"mix":2207},{"t":3179.233,"mix":2208},{"t":3180.289,"mix":2209},{"t":3181.428,"mix":2210},{"t":3182.477,"mix":2211},{"t":3183.624,"mix":2212},{"t":3184.679,"mix":2213},{"t":3185.807,"mix":2214},{"t":3186.875,"mix":2215},{"t":3188.022,"mix":2216},{"t":3189.021,"mix":2217},{"t":3189.947,"mix":2218},{"t":3190.953,"mix":2219},{"t":3191.924,"mix":2220},{"t":3192.852,"mix":2221},{"t":3193.77,"mix":2222},{"t":3194.731,"mix":2223},{"t":3195.701,"mix":2224},{"t":3196.719,"mix":2225},{"t":3197.754,"mix":2226},{"t":3198.703,"mix":2227},{"t":3199.667,"mix":2228},{"t":3200.575,"mix":2229},{"t":3201.524,"mix":2230},{"t":3202.459,"mix":2231},{"t":3203.422,"mix":2232},{"t":3204.347,"mix":2233},{"t":3205.285,"mix":2234},{"t":3206.266,"mix":2235},{"t":3207.182,"mix":2236},{"t":3208.08,"mix":2237},{"t":3209.027,"mix":2238},{"t":3209.963,"mix":2239},{"t":3210.888,"mix":2240},{"t":3211.811,"mix":2241},{"t":3212.778,"mix":2242},{"t":3213.718,"mix":2243},{"t":3214.67,"mix":2244},{"t":3215.619,"mix":2245},{"t":3216.562,"mix":2246},{"t":3217.467,"mix":2247},{"t":3218.4,"mix":2248},{"t":3219.345,"mix":2249},{"t":3220.292,"mix":2250},{"t":3221.23,"mix":2251},{"t":3222.138,"mix":2252},{"t":3223.047,"mix":2253},{"t":3223.928,"mix":2254},{"t":3224.901,"mix":2255},{"t":3225.798,"mix":2256},{"t":3226.728,"mix":2257},{"t":3227.678,"mix":2258},{"t":3228.722,"mix":2259},{"t":3229.575,"mix":2260},{"t":3230.459,"mix":2261},{"t":3231.456,"mix":2262},{"t":3232.337,"mix":2263},{"t":3233.262,"mix":2264},{"t":3234.176,"mix":2265},{"t":3235.07,"mix":2266},{"t":3235.973,"mix":2267},{"t":3236.882,"mix":2268},{"t":3238.822,"mix":2269},{"t":3239.854,"mix":2270},{"t":3243.395,"mix":2271},{"t":3246.977,"mix":2272},{"t":3250.258,"mix":2273},{"t":3254.057,"mix":2274},{"t":3257.476,"mix":2275},{"t":3260.828,"mix":2276},{"t":3264.326,"mix":2277},{"t":3267.774,"mix":2278},{"t":3270.974,"mix":2279},{"t":3274.046,"mix":2280},{"t":3277.175,"mix":2281},{"t":3279.977,"mix":2282},{"t":3282.645,"mix":2283},{"t":3285.585,"mix":2284},{"t":3288.691,"mix":2285},{"t":3291.671,"mix":2286},{"t":3295.288,"mix":2287},{"t":3298.719,"mix":2288},{"t":3301.755,"mix":2289},{"t":3305.164,"mix":2290},{"t":3308.453,"mix":2291},{"t":3311.765,"mix":2292},{"t":3315.336,"mix":2293},{"t":3318.952,"mix":2294},{"t":3322.308,"mix":2295},{"t":3325.564,"mix":2296},{"t":3328.518,"mix":2297},{"t":3331.558,"mix":2298},{"t":3334.778,"mix":2299},{"t":3337.823,"mix":2300},{"t":3341.273,"mix":2301},{"t":3345.784,"mix":2302},{"t":3350.269,"mix":2303},{"t":3354.565,"mix":2304},{"t":3358.357,"mix":2305},{"t":3363.362,"mix":2306},{"t":3367.379,"mix":2307},{"t":3371.608,"mix":2308},{"t":3375.762,"mix":2309},{"t":3379.73,"mix":2310},{"t":3384.208,"mix":2311},{"t":3387.962,"mix":2312},{"t":3392.037,"mix":2313},{"t":3395.992,"mix":2314},{"t":3399.726,"mix":2315},{"t":3403.456,"mix":2316},{"t":3407.189,"mix":2317},{"t":3410.817,"mix":2318},{"t":3414.79,"mix":2319},{"t":3418.472,"mix":2320},{"t":3422.635,"mix":2321},{"t":3426.375,"mix":2322},{"t":3429.939,"mix":2323},{"t":3433.122,"mix":2324},{"t":3436.747,"mix":2325},{"t":3440.719,"mix":2326},{"t":3444.155,"mix":2327},{"t":3447.718,"mix":2328},{"t":3451.303,"mix":2329},{"t":3459.549,"mix":2330},{"t":3461.282,"mix":2331},{"t":3462.754,"mix":2332},{"t":3463.98,"mix":2333},{"t":3465.189,"mix":2334},{"t":3466.409,"mix":2335},{"t":3467.705,"mix":2336},{"t":3468.921,"mix":2337},{"t":3470.161,"mix":2338},{"t":3471.49,"mix":2339},{"t":3472.81,"mix":2340},{"t":3474.084,"mix":2341},{"t":3475.347,"mix":2342},{"t":3476.548,"mix":2343},{"t":3477.817,"mix":2344},{"t":3479.016,"mix":2345},{"t":3480.204,"mix":2346},{"t":3481.565,"mix":2347},{"t":3482.794,"mix":2348},{"t":3484.064,"mix":2349},{"t":3485.235,"mix":2350},{"t":3486.502,"mix":2351},{"t":3487.672,"mix":2352},{"t":3488.897,"mix":2353},{"t":3490.091,"mix":2354},{"t":3491.329,"mix":2355},{"t":3492.663,"mix":2356},{"t":3493.92,"mix":2357},{"t":3495.16,"mix":2358},{"t":3496.386,"mix":2359},{"t":3497.671,"mix":2360},{"t":3498.818,"mix":2361},{"t":3500.106,"mix":2362},{"t":3501.426,"mix":2363},{"t":3502.603,"mix":2364},{"t":3503.883,"mix":2365},{"t":3505.201,"mix":2366},{"t":3506.425,"mix":2367},{"t":3507.712,"mix":2368},{"t":3508.897,"mix":2369},{"t":3510.193,"mix":2370},{"t":3511.429,"mix":2371},{"t":3512.616,"mix":2372},{"t":3513.988,"mix":2373},{"t":3515.166,"mix":2374},{"t":3516.414,"mix":2375},{"t":3517.647,"mix":2376},{"t":3518.866,"mix":2377},{"t":3520.085,"mix":2378},{"t":3521.327,"mix":2379},{"t":3522.586,"mix":2380},{"t":3523.828,"mix":2381},{"t":3525.059,"mix":2382},{"t":3526.3,"mix":2383},{"t":3527.505,"mix":2384},{"t":3528.7,"mix":2385},{"t":3529.983,"mix":2386},{"t":3531.253,"mix":2387},{"t":3532.531,"mix":2388},{"t":3533.772,"mix":2389},{"t":3535,"mix":2390},{"t":3536.178,"mix":2391},{"t":3537.441,"mix":2392},{"t":3538.709,"mix":2393},{"t":3540.007,"mix":2394},{"t":3541.318,"mix":2395},{"t":3542.625,"mix":2396},{"t":3543.789,"mix":2397},{"t":3545.046,"mix":2398},{"t":3546.284,"mix":2399},{"t":3547.534,"mix":2400},{"t":3548.754,"mix":2401},{"t":3549.933,"mix":2402},{"t":3551.234,"mix":2403},{"t":3552.419,"mix":2404},{"t":3553.701,"mix":2405},{"t":3554.955,"mix":2406},{"t":3556.22,"mix":2407},{"t":3557.809,"mix":2408},{"t":3559.01,"mix":2409},{"t":3560.285,"mix":2410},{"t":3561.627,"mix":2411},{"t":3562.849,"mix":2412},{"t":3564.073,"mix":2413},{"t":3565.378,"mix":2414},{"t":3566.714,"mix":2415},{"t":3567.878,"mix":2416},{"t":3569.135,"mix":2417},{"t":3570.331,"mix":2418},{"t":3571.601,"mix":2419},{"t":3572.847,"mix":2420},{"t":3574.222,"mix":2421},{"t":3575.795,"mix":2422},{"t":3577.164,"mix":2423},{"t":3578.736,"mix":2424},{"t":3580.15,"mix":2425},{"t":3581.928,"mix":2426},{"t":3583.411,"mix":2427},{"t":3584.857,"mix":2428},{"t":3586.221,"mix":2429},{"t":3587.902,"mix":2430},{"t":3589.583,"mix":2431},{"t":3591.095,"mix":2432},{"t":3592.545,"mix":2433},{"t":3594.157,"mix":2434},{"t":3595.805,"mix":2435},{"t":3597.705,"mix":2436},{"t":3599.819,"mix":2437},{"t":3601.886,"mix":2438},{"t":3607.716,"mix":2439},{"t":3609.151,"mix":2440},{"t":3610.232,"mix":2441},{"t":3611.296,"mix":2442},{"t":3612.353,"mix":2443},{"t":3613.557,"mix":2444},{"t":3614.664,"mix":2445},{"t":3615.642,"mix":2446},{"t":3616.658,"mix":2447},{"t":3617.855,"mix":2448},{"t":3618.825,"mix":2449},{"t":3620.054,"mix":2450},{"t":3621.059,"mix":2451},{"t":3622.155,"mix":2452},{"t":3623.188,"mix":2453},{"t":3624.352,"mix":2454},{"t":3625.532,"mix":2455},{"t":3626.571,"mix":2456},{"t":3627.639,"mix":2457},{"t":3628.734,"mix":2458},{"t":3629.88,"mix":2459},{"t":3631.008,"mix":2460},{"t":3632.098,"mix":2461},{"t":3633.164,"mix":2462},{"t":3634.357,"mix":2463},{"t":3635.416,"mix":2464},{"t":3636.529,"mix":2465},{"t":3637.668,"mix":2466},{"t":3638.777,"mix":2467},{"t":3639.854,"mix":2468},{"t":3640.928,"mix":2469},{"t":3642.037,"mix":2470},{"t":3643.09,"mix":2471},{"t":3644.191,"mix":2472},{"t":3645.284,"mix":2473},{"t":3646.387,"mix":2474},{"t":3647.409,"mix":2475},{"t":3648.52,"mix":2476},{"t":3649.597,"mix":2477},{"t":3650.655,"mix":2478},{"t":3651.624,"mix":2479},{"t":3652.622,"mix":2480},{"t":3653.63,"mix":2481},{"t":3654.677,"mix":2482},{"t":3655.626,"mix":2483},{"t":3656.806,"mix":2484},{"t":3657.808,"mix":2485},{"t":3658.854,"mix":2486},{"t":3664.63,"mix":2487},{"t":3670.56,"mix":2488},{"t":3675.546,"mix":2489},{"t":3681.15,"mix":2490},{"t":3682.329,"mix":2491},{"t":3683.267,"mix":2492},{"t":3684.31,"mix":2493},{"t":3685.24,"mix":2494},{"t":3686.307,"mix":2495},{"t":3687.262,"mix":2496},{"t":3688.321,"mix":2497},{"t":3689.358,"mix":2498},{"t":3690.315,"mix":2499},{"t":3691.307,"mix":2500},{"t":3692.382,"mix":2501},{"t":3693.317,"mix":2502},{"t":3694.359,"mix":2503},{"t":3695.43,"mix":2504},{"t":3696.437,"mix":2505},{"t":3697.397,"mix":2506},{"t":3698.665,"mix":2507},{"t":3699.655,"mix":2508},{"t":3703.577,"mix":2509},{"t":3708.394,"mix":2510},{"t":3713.397,"mix":2511},{"t":3718.025,"mix":2512},{"t":3722.634,"mix":2513},{"t":3727.562,"mix":2514},{"t":3731.944,"mix":2515},{"t":3736.529,"mix":2516},{"t":3741.834,"mix":2517},{"t":3748.954,"mix":2518},{"t":3755.859,"mix":2519},{"t":3757.577,"mix":2520},{"t":3758.604,"mix":2521},{"t":3759.474,"mix":2522},{"t":3760.317,"mix":2523},{"t":3761.167,"mix":2524},{"t":3761.902,"mix":2525},{"t":3762.613,"mix":2526},{"t":3763.266,"mix":2527},{"t":3764.078,"mix":2528},{"t":3764.802,"mix":2529},{"t":3765.494,"mix":2530},{"t":3766.199,"mix":2531},{"t":3766.884,"mix":2532},{"t":3767.575,"mix":2533},{"t":3768.225,"mix":2534},{"t":3768.905,"mix":2535},{"t":3769.589,"mix":2536},{"t":3770.278,"mix":2537},{"t":3771.006,"mix":2538},{"t":3771.768,"mix":2539},{"t":3772.506,"mix":2540},{"t":3773.229,"mix":2541},{"t":3773.959,"mix":2542},{"t":3774.609,"mix":2543},{"t":3775.325,"mix":2544},{"t":3776.039,"mix":2545},{"t":3776.763,"mix":2546},{"t":3777.468,"mix":2547},{"t":3778.198,"mix":2548},{"t":3778.895,"mix":2549},{"t":3779.576,"mix":2550},{"t":3780.276,"mix":2551},{"t":3780.939,"mix":2552},{"t":3781.655,"mix":2553},{"t":3782.308,"mix":2554},{"t":3783.016,"mix":2555},{"t":3783.711,"mix":2556},{"t":3784.42,"mix":2557},{"t":3785.115,"mix":2558},{"t":3785.83,"mix":2559},{"t":3786.54,"mix":2560},{"t":3787.29,"mix":2561},{"t":3787.974,"mix":2562},{"t":3788.705,"mix":2563},{"t":3789.425,"mix":2564},{"t":3790.202,"mix":2565},{"t":3790.889,"mix":2566},{"t":3791.565,"mix":2567},{"t":3792.3,"mix":2568},{"t":3793.002,"mix":2569},{"t":3793.661,"mix":2570},{"t":3794.406,"mix":2571},{"t":3795.061,"mix":2572},{"t":3795.817,"mix":2573},{"t":3796.505,"mix":2574},{"t":3797.236,"mix":2575},{"t":3797.966,"mix":2576},{"t":3798.717,"mix":2577},{"t":3799.42,"mix":2578},{"t":3800.085,"mix":2579},{"t":3800.813,"mix":2580},{"t":3801.501,"mix":2581},{"t":3802.198,"mix":2582},{"t":3802.901,"mix":2583},{"t":3803.659,"mix":2584},{"t":3804.392,"mix":2585},{"t":3805.078,"mix":2586},{"t":3805.739,"mix":2587},{"t":3806.468,"mix":2588},{"t":3807.215,"mix":2589},{"t":3807.944,"mix":2590},{"t":3808.701,"mix":2591},{"t":3809.459,"mix":2592},{"t":3815.61,"mix":2593},{"t":3821.203,"mix":2594},{"t":3827.427,"mix":2595},{"t":3833.773,"mix":2596},{"t":3834.38,"mix":2597},{"t":3835.078,"mix":2598},{"t":3835.74,"mix":2599},{"t":3836.434,"mix":2600},{"t":3837.086,"mix":2601},{"t":3837.749,"mix":2602},{"t":3838.421,"mix":2603},{"t":3839.024,"mix":2604},{"t":3839.723,"mix":2605},{"t":3840.394,"mix":2606},{"t":3841.057,"mix":2607},{"t":3841.643,"mix":2608},{"t":3842.26,"mix":2609},{"t":3842.909,"mix":2610},{"t":3843.555,"mix":2611},{"t":3844.18,"mix":2612},{"t":3844.877,"mix":2613},{"t":3845.48,"mix":2614},{"t":3846.091,"mix":2615},{"t":3846.814,"mix":2616},{"t":3854.371,"mix":2617},{"t":3856.371,"mix":2611},{"t":3858.371,"mix":2612},{"t":3860.371,"mix":2613},{"t":3862.371,"mix":2614},{"t":3864.371,"mix":2615},{"t":3866.371,"mix":2616},{"t":3868.371,"mix":2617}]
'''  # Use the full JSON
timestamps1 = json.loads(timestamps_json)

def parse_start_time_from_url(youtube_url):
    parsed_url = urlparse.urlparse(youtube_url)
    query_params = urlparse.parse_qs(parsed_url.query)
    start_time = query_params.get('t', ['0'])[0]  # Default to '0' if not provided
    if 's' in start_time:
        # Extract time in seconds from the URL parameter
        time_seconds = int(re.search(r'\d+', start_time).group())
        return time_seconds
    else:
        return int(start_time)

def download_audio_with_retries(youtube_url, output_path, max_retries=5):
    retry_count = 0
    while retry_count < max_retries:
        try:
            yt = YouTube(youtube_url)
            audio_stream = yt.streams.filter(only_audio=True).first()
            if not audio_stream:
                print("No audio stream found.")
                return None
            # Download the audio stream directly without conversion
            downloaded_file = audio_stream.download(filename=output_path)
            return downloaded_file
        except Exception as e:
            print(f"Attempt {retry_count + 1} failed: {str(e)}")
            time.sleep(5)  # wait 5 seconds before retrying
            retry_count += 1
    print("Failed to download after several retries.")
    return None

def convert_time_to_seconds(time):
    if isinstance(time, str) and ':' in time:
        minutes, seconds = map(int, time.split(':'))
        return minutes * 60 + seconds
    elif isinstance(time, (int, float)):
        return time
    else:
        raise ValueError("Time format must be a string 'MM:SS' or a number representing seconds")

def convert_to_ogg(input_path, output_path, start_time=0, end_time=None):
    try:
        # Convert start time to seconds and add offset
        start_total_seconds = convert_time_to_seconds(start_time) + 0.15
        
        # Initialize the ffmpeg command
        command = [
            'ffmpeg', '-ss', str(start_total_seconds), '-i', input_path,
            '-c:a', 'libvorbis', '-q:a', '5'
        ]
        
        if end_time is not None:
            # Convert end time to seconds
            end_total_seconds = convert_time_to_seconds(end_time)
            # Calculate the duration of the clip
            clip_duration = end_total_seconds - start_total_seconds
            command.extend(['-t', str(clip_duration)])
        
        command.append(output_path)
        
        subprocess.run(command, check=True)
        return output_path
    except subprocess.CalledProcessError as e:
        print(f"Error during conversion: {e}")
        return None
    except ValueError as e:
        print(f"Invalid time format: {e}")
        return None

def load_audio_segment(audio_path, start_time=0, end_time=None, sr=11025):
    start_total_seconds = convert_time_to_seconds(start_time)
    
    if end_time is not None:
        end_total_seconds = convert_time_to_seconds(end_time)
        duration = end_total_seconds - start_total_seconds
    else:
        # If end_time is None, calculate the full length of the audio
        full_duration = librosa.get_duration(filename=audio_path)
        duration = full_duration - start_total_seconds

    y, sr = librosa.load(audio_path, sr=sr, offset=start_total_seconds, duration=duration)
    return y, sr

# URLs for YouTube videos
youtube_url1 = 'https://www.youtube.com/watch?v=O3MVY6UiMag'
youtube_url2 = 'https://www.youtube.com/watch?v=7eEekvDKhCw&t=8s'

start_time1 = 0.40
start_time2 = parse_start_time_from_url(youtube_url2)
end_time1 = "64:09"
end_time2 = "72:48"
full_duration1 = 0
full_duration2 = 0

# Download audio files
audio_path1 = download_audio_with_retries(youtube_url1, 'youtube_audio1.mp4')
if audio_path1:
    print('audio_path1 downloaded')
audio_path2 = download_audio_with_retries(youtube_url2, 'youtube_audio2.mp4')
if audio_path2:
    print('audio_path2 downloaded')


# Convert to OGG
ogg_path1 = "youtube_audio1.ogg"
ogg_path2 = "youtube_audio2.ogg"
#convert_to_ogg(audio_path1, ogg_path1, start_time1, end_time1)
convert_to_ogg(audio_path2, ogg_path2, start_time2, end_time2)

audio_path1 downloaded
audio_path2 downloaded


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

'youtube_audio2.ogg'

In [2]:
import gc
from concurrent.futures import ThreadPoolExecutor

HOP_LENGTH = 128
HOP_LENGTH_1 = 1024
HOP_LENGTH_2 = 256
OVERLAP_HOP = 256

def load_and_preprocess_audio_ORIGINAL(audio_path, target_sr=11025):
    # Load audio file at a reduced sample rate
    y, sr = librosa.load(audio_path, sr=target_sr)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256)
    log_S = librosa.power_to_db(S, ref=np.max)
    return log_S.T, sr

def load_and_preprocess_audio(audio_path, target_sr=11025, hop_length=HOP_LENGTH):
    # Load audio file at a higher sample rate for better temporal resolution
    y, sr = librosa.load(audio_path, sr=target_sr)
    
    # Harmonic-Percussive Source Separation (HPSS) --- cutting for resource savings
    #y_harmonic, y_percussive = librosa.effects.hpss(y)
    
    # Mel-spectrogram for harmonic component
    #S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=64, hop_length=HOP_LENGTH)
    #log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
    
    # Constant-Q Transform for better frequency resolution
    #CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

    # Combine features
    #combined_features = np.vstack((log_S_harmonic, CQT))
    #print(f"Combined features shape: {combined_features.shape}")
    
    #return combined_features.T, sr

    #BELOW IS STRIPPED VERSION OF ABOVE
    y, sr = librosa.load(audio_path, sr=sr)
    
    # Mel-spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=512, hop_length=HOP_LENGTH)
    log_S = librosa.power_to_db(S, ref=np.max)
    
    # Constant-Q Transform
    CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

    combined_features = np.vstack((log_S, CQT))
    return combined_features.T, sr


def load_and_preprocess_audio_INCREMENTAL(audio_path, target_sr=11025, chunk_duration=10):
    # Load the audio file in chunks
    y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = num_chunks // 10  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Harmonic-Percussive Source Separation (HPSS)
        y_harmonic, y_percussive = librosa.effects.hpss(y_chunk)
        
        # Mel-spectrogram for harmonic component
        S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=256)
        log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
        
        # Constant-Q Transform for better frequency resolution
        CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr)), ref=np.max)
        
        # Combine features
        combined_chunk_features = np.vstack((log_S_harmonic, CQT))
        
        combined_features.append(combined_chunk_features.T)
        
        # Print progress
        if (i + 1) % progress_step == 0:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    
    return combined_features, sr

def load_and_preprocess_audio_OVERLAP(audio_path, target_sr=11025, chunk_duration=600, n_mels=256):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)
    num_chunks = (len(y) + chunk_length - 1) // chunk_length

    combined_features = []
    progress_step = max(1, num_chunks // 10)
    
    num_overlaps = HOP_LENGTH // OVERLAP_HOP
    print(f"Creating {num_overlaps} overlapping windows per chunk")
    
    print(f"Processing file: {audio_path}")
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Overlapping windows
        chunk_features = []
        for offset in range(0, HOP_LENGTH, OVERLAP_HOP):
            if offset > 0:
                y_shifted = np.pad(y_chunk, (offset, 0), mode='constant')[:-offset]
            else:
                y_shifted = y_chunk
            
            # HPSS
            y_harmonic, y_percussive = librosa.effects.hpss(y_shifted)
            
            # Mel-spectrogram
            S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=n_mels, hop_length=HOP_LENGTH)
            log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
            
            # Constant-Q Transform
            CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_shifted, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)
            
            # Combine features for this offset
            combined_chunk_features = np.vstack((log_S_harmonic, CQT))
            chunk_features.append(combined_chunk_features.T)
            del y_shifted, y_harmonic, y_percussive, S_harmonic, log_S_harmonic, CQT
            gc.collect()

        combined_chunk_features = np.concatenate(chunk_features, axis=1)
        combined_features.append(combined_chunk_features)
        del chunk_features
        gc.collect()
        
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

#low and high averageed hoplengths
def load_and_preprocess_audio_AVERAGE(audio_path, target_sr=11025, chunk_duration=600, n_mels=256):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)
    num_chunks = (len(y) + chunk_length - 1) // chunk_length

    combined_features = []
    progress_step = max(1, num_chunks // 10)
    
    num_overlaps = HOP_LENGTH_2 // OVERLAP_HOP
    print(f"Creating {num_overlaps} overlapping windows per chunk")
    
    print(f"Processing file: {audio_path}")
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Overlapping windows for both hop lengths
        chunk_features_list = []
        max_length = 0
        for hop_length in [HOP_LENGTH_1, HOP_LENGTH_2]:
            chunk_features = []
            for offset in range(0, hop_length, OVERLAP_HOP):
                if offset > 0:
                    y_shifted = np.pad(y_chunk, (offset, 0), mode='constant')[:-offset]
                else:
                    y_shifted = y_chunk
                
                # Mel-spectrogram
                S = librosa.feature.melspectrogram(y=y_shifted, sr=sr, n_mels=n_mels, hop_length=hop_length)
                log_S = librosa.power_to_db(S, ref=np.max)
                
                chunk_features.append(log_S.T)
                del y_shifted, S, log_S
                gc.collect()

            # Find the maximum length of the feature matrices
            max_length = max(max_length, max(f.shape[0] for f in chunk_features))
            chunk_features_list.append(chunk_features)
            del chunk_features
            gc.collect()
        
        # Resize all feature matrices to the maximum length and average
        resized_chunk_features_list = []
        for features in chunk_features_list:
            resized_features = [resize(f, (max_length, f.shape[1]), anti_aliasing=True) for f in features]
            averaged_chunk_features = np.mean(resized_features, axis=0)
            resized_chunk_features_list.append(averaged_chunk_features)
        
        # Average the features from different hop lengths
        combined_chunk_features = np.mean(resized_chunk_features_list, axis=0)
        combined_features.append(combined_chunk_features)
        del chunk_features_list, resized_chunk_features_list
        gc.collect()
        
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

def load_and_preprocess_audio_INCREMENTAL_NOHPSS(audio_path, target_sr=11025, chunk_duration=10, n_mels=256, mel_weight=1.0, cqt_weight=1.0, tempo_weight=1.0):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = num_chunks // 10  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Mel-spectrogram
        S = librosa.feature.melspectrogram(y=y_chunk, sr=sr, n_mels=n_mels, hop_length=HOP_LENGTH)
        log_S = librosa.power_to_db(S, ref=np.max)
        
        # Constant-Q Transform
        CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

        # Tempogram
        onset_env = librosa.onset.onset_strength(y=y_chunk, sr=sr, hop_length=HOP_LENGTH)
        tempogram = librosa.feature.tempogram(onset_envelope=onset_env, sr=sr, hop_length=HOP_LENGTH)

        # Ensure the same length for both arrays
        min_length = min(log_S.shape[1], CQT.shape[1])
        log_S = log_S[:, :min_length]
        CQT = CQT[:, :min_length]
        
        # Normalize features
        log_S = normalize_features(log_S)
        CQT = normalize_features(CQT)

        log_S *= mel_weight
        CQT *= cqt_weight
        tempogram *= tempo_weight
        
        # Combine features
        combined_chunk_features = np.vstack((log_S, CQT))

        combined_features.append(combined_chunk_features.T)
        del log_S, CQT, y_chunk, S, combined_chunk_features
        gc.collect()
        
        # Print progress
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

def process_chunk(y_chunk, sr, hop_length):
    # Chroma feature extraction
    chroma = librosa.feature.chroma_stft(y=y_chunk, sr=sr, hop_length=HOP_LENGTH)
    # chroma = normalize_features(chroma)
    
    # Constant-Q Transform feature extraction
    CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)
    # CQT = normalize_features(CQT)
    
    # Ensure the same length for both arrays
    min_length = min(chroma.shape[1], CQT.shape[1])
    chroma = chroma[:, :min_length]
    CQT = CQT[:, :min_length]
    
    # Combine features
    combined_chunk_features = np.vstack((chroma, CQT)).T
    
    return combined_chunk_features

def load_and_preprocess_audio_combined(audio_path, target_sr=44100, chunk_duration=10, hop_length=HOP_LENGTH):
    y, sr = librosa.load(audio_path, sr=target_sr)
    y = normalize_audio(y)  # Normalize the raw audio signal
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = max(1, num_chunks // 10)  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")

    def process_and_collect(i):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        chunk_features = process_chunk(y_chunk, sr, HOP_LENGTH)
        return chunk_features

    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_and_collect, i) for i in range(num_chunks)]
        for i, future in enumerate(futures):
            combined_features.append(future.result())
            # Print progress
            if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
                print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")
            gc.collect()

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

# Normalize the raw audio signal
def normalize_audio(y, epsilon=1e-8):
    return (y - np.mean(y)) / (np.std(y) + epsilon)
    
def normalize_features(features, epsilon=1e-8):
    mean = np.mean(features, axis=0)
    std_dev = np.std(features, axis=0)
    return (features - mean) / (std_dev + epsilon)

    
# Load and preprocess both audio recordings in OGG format

S1, sr1 = load_and_preprocess_audio_combined(ogg_path1)
#S1, sr1 = load_and_preprocess_audio_INCREMENTAL_NOHPSS(ogg_path1, mel_weight=3.0, cqt_weight=1.0, tempo_weight=2.0)

Total chunks: 385, Progress step: 38
Processed 10% of chunks
Processed 20% of chunks
Processed 30% of chunks
Processed 39% of chunks
Processed 49% of chunks
Processed 59% of chunks
Processed 69% of chunks
Processed 79% of chunks
Processed 89% of chunks
Processed 99% of chunks
Processed 100% of chunks
Combined features shape: (1326176, 96)


In [17]:
#making own step so no need to re-run audio1 for multiple recordings.

S2, sr2 = load_and_preprocess_audio_combined(ogg_path2)
#S2, sr2 = load_and_preprocess_audio_INCREMENTAL_NOHPSS(ogg_path2, mel_weight=3.0, cqt_weight=1.0, tempo_weight=2.0)

Total chunks: 436, Progress step: 43
Processed 10% of chunks
Processed 20% of chunks
Processed 30% of chunks
Processed 39% of chunks
Processed 49% of chunks
Processed 59% of chunks
Processed 69% of chunks
Processed 79% of chunks
Processed 89% of chunks
Processed 99% of chunks
Processed 100% of chunks
Combined features shape: (1502404, 96)


In [18]:
print(S1.shape,S2.shape)

(1326176, 96) (1502404, 96)


In [ ]:
from fastdtw import fastdtw
#from dtaidistance import dtw
    
def dynamic_time_warping_approx(S1, S2):
    distance, path = fastdtw(S1, S2)
    return path

#def constrained_dtw(S1, S2):
#    # Calculate the DTW distance with a window constraint
#    distance, paths = dtw.warping_paths(S1, S2, window=1000)
#    # Find the best path
#    path = dtw.best_path(paths)
#    return path

#warping_path = constrained_dtw(S1, S2)
warping_path = dynamic_time_warping_approx(S1, S2)

In [ ]:
def adjust_timestamps(wp, timestamps, sr):
    mapping = {row[0]: row[1] for row in wp}
    adjusted_timestamps = []
    
    for entry in timestamps:
        original_frame = int((entry['t']) * sr / HOP_LENGTH)
        if original_frame in mapping:
            adjusted_time = mapping[original_frame] * HOP_LENGTH / sr
            adjusted_timestamps.append({"t": adjusted_time, "mix": entry['mix']})
    
    # Ensure the last timestamp is included and set "t" to 9999
    if timestamps:
        last_entry = timestamps[-1]
        last_entry_adjusted = {"t": 9999, "mix": last_entry['mix']}
        if adjusted_timestamps and adjusted_timestamps[-1]['mix'] == last_entry['mix']:
            adjusted_timestamps[-1] = last_entry_adjusted
        else:
            adjusted_timestamps.append(last_entry_adjusted)
    
    return adjusted_timestamps

# Sample JSON timestamps for the first recording (assumed already loaded)
adjusted_timestamps = adjust_timestamps(warping_path, timestamps1, sr1)

In [ ]:
def calculate_ratios(timestamps):
    ratios = []
    for i in range(1, len(timestamps)):
        current_ratio = abs(timestamps[i]['t'] - timestamps[i-1]['t']) if timestamps[i-1]['t'] != 0 else 0
        ratios.append(current_ratio)
    return ratios

def compare_and_flag_changes(adjusted_timestamps, original_timestamps, audio_length, neighbor_count=5):
    # Set last timestamp as per new requirement
    adjusted_timestamps[-1]['t'] = audio_length + 1

    # Calculate differences and ratios
    adjusted_ratios = calculate_ratios(adjusted_timestamps)
    original_ratios = calculate_ratios(original_timestamps)

    # Array to hold timestamps that are significantly different
    flagged_timestamps = []

    # Analyze ratios for significant changes
    for i in range(len(adjusted_ratios) - 1):  # Ignore the last timestamp in comparison
        start = max(0, i - neighbor_count)
        end = min(len(original_ratios) - 1, i + neighbor_count + 1)  # Avoid including the last in comparison
        
        # Calculate neighborhood average without including out-of-range values
        neighborhood_original = original_ratios[start:end]
        if not neighborhood_original:
            continue
        neighborhood_average = np.mean(neighborhood_original)
        
        # Check if the current adjusted ratio is significantly different
        if adjusted_ratios[i] > 1.5 * neighborhood_average:
            flagged_timestamps.append({
                "mix": adjusted_timestamps[i]['mix'],
                "original_ratio": original_ratios[i] if i < len(original_ratios) else 0,
                "adjusted_ratio": adjusted_ratios[i],
                "average_neighbors": neighborhood_average
            })

    return flagged_timestamps

# Adjust so that the first timestamp is zero
initial_offset = -adjusted_timestamps[0]['t']

# Initialize an empty list to store the new adjusted timestamps
new_adjusted_timestamps = []
previous_t = None  # Variable to hold the previous timestamp

for item in adjusted_timestamps:
    adjusted_t = round(item["t"] + initial_offset, 3)
    new_adjusted_timestamps.append({"t": adjusted_t, "mix": item["mix"]})
    
    # Check if the previous timestamp is defined and compare the current timestamp with the previous one
    if previous_t is not None and (adjusted_t - previous_t < 0.42):
        difference = adjusted_t - previous_t
        print(f"Close timestamps found: Mix: {item['mix']}, Difference: {difference:.3f}, Previous - {previous_t}, Current - {adjusted_t}")
    
    # Update the previous_t to the current timestamp for the next iteration
    previous_t = adjusted_t


# Print the adjusted timestamps and initial offset
print(json.dumps(new_adjusted_timestamps, indent=4))
# Here we adjust to show the total offset from the original video start
full_offset = abs(initial_offset) + abs(start_time2)
print(f"Total Offset from Video Start: {full_offset}, initial {initial_offset} + start_time2 {start_time2}")
print(youtube_url2)

def calculate_full_duration(audio_path, start_time, end_time):
    start_total_seconds = convert_time_to_seconds(start_time)
    
    if end_time is not None:
        end_total_seconds = convert_time_to_seconds(end_time)
    else:
        # Calculate the full length of the audio if end_time is None
        full_duration = librosa.get_duration(path=audio_path)
        end_total_seconds = full_duration

    full_duration = end_total_seconds - start_total_seconds
    return full_duration

full_duration2 = calculate_full_duration(ogg_path2, start_time2, end_time2)

#Flag anything that exceeds 50% difference compared to neighboring measures.
flagged_timestamps = compare_and_flag_changes(new_adjusted_timestamps, timestamps1, full_duration2)
print("Flagged Timestamps:")
for ft in flagged_timestamps:
    print(f"Mix: {ft['mix']}, Original Ratio: {ft['original_ratio']:.3f}, Adjusted Ratio: {ft['adjusted_ratio']:.3f}, Neighbors' Avg.: {ft['average_neighbors']:.3f}")

# Cleanup downloaded and converted files
#os.remove(audio_path1)
os.remove(audio_path2)
#os.remove(ogg_path1)
os.remove(ogg_path2)